In [ ]:
# Cell 0
# import pipeline dependencies

import os
import gc
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

In [ ]:
# Cell 1
# mount Drive; load cached df_sample or rebuild from raw parquets for the 3 target O-D pairs

from google.colab import drive
drive.mount('/drive')

DRIVE_PATH    = '/drive/MyDrive/flight-project/df_sample.csv'
VAL_CACHE     = '/drive/MyDrive/flight-project/df_val_egll_lgav.csv'
RAW_FIR       = '/drive/MyDrive/flight-project/Final_Wide_Report.parquet'
RAW_FLIGHTS   = '/drive/MyDrive/flight-project/Flights_20230901_20230930.parquet'
TARGET_OD     = [('LEBL', 'LEPA'), ('EGLL', 'KJFK'), ('LPPT', 'EDDB')]
VAL_OD        = ('EGLL', 'LGAV')
force_rebuild = False

if os.path.exists(DRIVE_PATH) and not force_rebuild:
    df_sample = pd.read_csv(DRIVE_PATH)
    print(f'Loaded from Drive: {df_sample.shape}')
else:
    print('Rebuilding from raw parquets...')
    fir     = pd.read_parquet(RAW_FIR)
    flights = pd.read_parquet(RAW_FLIGHTS)
    flights = flights[flights['ICAO Flight Type'] == 'S'].drop(columns=['STATFOR Market Segment'])

    df = fir.merge(flights, on='ECTRL ID', how='inner', suffixes=('', '_drop'))
    del fir, flights   # free ~4 GB before any further work
    df.drop(columns=[c for c in df.columns if c.endswith('_drop')], inplace=True)

    od_idx    = pd.MultiIndex.from_frame(df[['ADEP', 'ADES']])
    df_sample = df[od_idx.isin(TARGET_OD)].reset_index(drop=True)
    df_val_   = df[od_idx.isin([VAL_OD])].reset_index(drop=True)
    del df   # free the full join - both subsets are extracted

    os.makedirs(os.path.dirname(DRIVE_PATH), exist_ok=True)
    df_sample.to_csv(DRIVE_PATH, index=False)
    df_val_.to_csv(VAL_CACHE, index=False)
    del df_val_
    print(f'Training sample saved: {df_sample.shape}')
    print(f'Validation sample saved: {VAL_OD[0]}-{VAL_OD[1]}')

df_sample.head(2)

In [ ]:
# Cell 2
# identify which FIRs are active per O-D pair (>=25% of flights must cross), union across all pairs into ACTIVE_FIRS
# uses FIR/UIR suffix, not dtype==float64 -- the dtype filter also matches lat/lon and Requested FL,
# which corrupts both the KMeans distance metric and merge_similar_clusters' centroid comparison

_fir_cols = [c for c in df_sample.columns if c.endswith(('FIR', 'UIR'))]

ACTIVE_FIRS = []
for (adep, ades), grp in df_sample.groupby(['ADEP', 'ADES']):
    pair_firs = [f for f in _fir_cols if (grp[f] > 0).mean() >= 0.25]
    ACTIVE_FIRS.extend(pair_firs)
    print(f'{adep}-{ades}: {len(pair_firs)} active FIRs - {pair_firs}')

ACTIVE_FIRS = sorted(set(ACTIVE_FIRS))
print(f'\nACTIVE_FIRS ({len(ACTIVE_FIRS)}): {ACTIVE_FIRS}')


In [ ]:
# Cell 3
# binary route signature: 1 if a flight crossed a given FIR, 0 if not - input to all clustering

def make_route_signatures(df, fir_cols):
    return (df[fir_cols] > 0).astype(int)

In [ ]:
# Cell 4
# L1:   O-D pair (groupby)
# L1.5: hard group by binary FIR signature (which FIRs crossed) -- explicit, not KMeans
# L2:   KMeans on actual FIR distances within each L1.5 signature group
# L3:   group by AC type within each L2 cluster, pooled fallback when a type has too few
#       flights in that cluster (see full_summary_ac in cell 34, l3_centroids in cell 38)

import warnings
from sklearn.exceptions import ConvergenceWarning

MIN_CLUSTER_SIZE = 15

def _best_k(X, k_range=range(2, 6)):
    scores = {}
    for k in k_range:
        if len(X) <= k:
            continue
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', ConvergenceWarning)
            labels = KMeans(n_clusters=k, random_state=158, n_init=10).fit_predict(X)
        if len(set(labels)) < k:
            continue
        if pd.Series(labels).value_counts().min() < MIN_CLUSTER_SIZE:
            continue
        scores[k] = silhouette_score(X, labels)
    return max(scores, key=scores.get) if scores else 1


def cluster_od_kmeans(df, fir_cols):
    sigs   = make_route_signatures(df, fir_cols)
    labels = pd.Series(-1, index=df.index, dtype=int)
    for (adep, ades), grp in df.groupby(['ADEP', 'ADES']):
        X = sigs.loc[grp.index].values
        k = _best_k(X)
        grp_labels = (
            np.zeros(len(grp), dtype=int) if k == 1
            else KMeans(n_clusters=k, random_state=158, n_init=10).fit_predict(X)
        )
        labels.loc[grp.index] = grp_labels
        counts = pd.Series(grp_labels).value_counts().sort_index().to_dict()
        print(f'{adep}-{ades} | k={k} | {counts}')
    return labels


def cluster_od_3layer(df, fir_cols):
    from sklearn.preprocessing import StandardScaler
    labels_l15   = pd.Series(-1, index=df.index, dtype=int)
    labels_final = pd.Series(-1, index=df.index, dtype=int)

    for (adep, ades), od_grp in df.groupby(['ADEP', 'ADES']):
        # L1.5: hard group by binary FIR signature -- which FIRs were crossed
        sig_series = (od_grp[fir_cols] > 0).astype(int).apply(tuple, axis=1)

        cluster_counter = 0
        sig_counter     = 0
        for sig_val, sig_grp in od_grp.groupby(sig_series):
            labels_l15.loc[sig_grp.index] = sig_counter
            sig_counter += 1

            # L2: KMeans on actual FIR distances within this signature group
            sub_X = StandardScaler().fit_transform(sig_grp[fir_cols].fillna(0).values)
            k2    = _best_k(sub_X)
            l2    = (np.zeros(len(sig_grp), dtype=int) if k2 == 1
                     else KMeans(n_clusters=k2, random_state=158, n_init=10).fit_predict(sub_X))

            for l2_id in sorted(set(l2)):
                labels_final.loc[sig_grp.index[l2 == l2_id]] = cluster_counter
                cluster_counter += 1

            counts = {j: int((l2 == j).sum()) for j in sorted(set(l2))}
            n_firs = sum(sig_val)
            print(f'{adep}-{ades} | sig({n_firs} FIRs) k2={k2} | {counts}')

    return labels_l15, labels_final

In [ ]:
# Cell 5
# DBSCAN per O-D pair using Hamming distance on binary signatures - kept for comparison; KMeans is primary

def cluster_od_dbscan(df, fir_cols, eps=0.3, min_samples=3):
    # eps on [0,1] hamming scale: 0.3 = flights differing in <=5/15 FIRs are neighbours
    sigs   = make_route_signatures(df, fir_cols)
    labels = pd.Series(-1, index=df.index, dtype=int)
    for (adep, ades), grp in df.groupby(['ADEP', 'ADES']):
        X   = sigs.loc[grp.index].values
        raw = DBSCAN(eps=eps, min_samples=min_samples, metric='hamming').fit_predict(X)
        labels.loc[grp.index] = raw
        n_clusters = len(set(raw)) - (1 if -1 in raw else 0)
        n_noise    = (raw == -1).sum()
        counts     = pd.Series(raw).value_counts().sort_index().to_dict()
        print(f'{adep}-{ades} | clusters={n_clusters}  noise={n_noise} | {counts}')
    return labels

In [ ]:
# Cell 6
# run 3-layer clustering and DBSCAN comparison

print('3-Layer KMeans (L2: binary FIR, L3: distance)')
kmeans_l2, kmeans_labels = cluster_od_3layer(df_sample, ACTIVE_FIRS)

print('\nDBSCAN (eps=0.3, min_samples=3) - comparison')
dbscan_labels = cluster_od_dbscan(df_sample, ACTIVE_FIRS)

In [ ]:
# Cell 7
# attach cluster labels back to df_sample and print the full distribution

df_sample['cluster_l2']     = kmeans_l2
df_sample['cluster_kmeans'] = kmeans_labels
df_sample['cluster_dbscan'] = dbscan_labels

print(df_sample[['ADEP', 'ADES', 'cluster_l2', 'cluster_kmeans', 'cluster_dbscan']].value_counts().sort_index())

In [ ]:
# Cell 7b
# post-clustering merge: collapse cluster pairs with near-identical FIR distance profiles
# L2 norm between cluster mean FIR vectors < MERGE_THRESHOLD_NM -> merge smaller into larger
# also force-merges any cluster below MIN_CLUSTER_SIZE into its nearest neighbour regardless
# of threshold -- L1.5's hard FIR-signature split can produce clusters of n=1-4 that KMeans'
# own MIN_CLUSTER_SIZE guard never touches, since that guard only governs splits KMeans itself
# makes, not the signature groups formed before KMeans ever runs

MERGE_THRESHOLD_NM = 100.0

def merge_similar_clusters(df, fir_cols, label_col='cluster_kmeans',
                            threshold=MERGE_THRESHOLD_NM, min_size=MIN_CLUSTER_SIZE):
    df = df.copy()
    changed = True
    while changed:
        changed = False
        for (adep, ades), grp in df.groupby(['ADEP', 'ADES']):
            clusters = sorted(grp[label_col].unique())
            if len(clusters) < 2:
                continue
            centroids = {c: grp.loc[grp[label_col] == c, fir_cols].mean().values for c in clusters}
            sizes     = {c: int((grp[label_col] == c).sum()) for c in clusters}

            undersized = [c for c in clusters if sizes[c] < min_size]
            if undersized:
                src = min(undersized, key=lambda c: sizes[c])
                dst = min((c for c in clusters if c != src),
                          key=lambda c: np.linalg.norm(centroids[src] - centroids[c]))
                dist = np.linalg.norm(centroids[src] - centroids[dst])
                df.loc[(df['ADEP'] == adep) & (df['ADES'] == ades) & (df[label_col] == src), label_col] = dst
                print(f'{adep}-{ades}: C{src} (n={sizes[src]}, below min_size={min_size}) -> '
                      f'C{dst} (n={sizes[dst]})  [forced, L2={dist:.1f} nm]')
                changed = True
                break

            min_dist, a, b = np.inf, None, None
            for i, ci in enumerate(clusters):
                for cj in clusters[i+1:]:
                    d = np.linalg.norm(centroids[ci] - centroids[cj])
                    if d < min_dist:
                        min_dist, a, b = d, ci, cj
            if min_dist < threshold:
                na, nb = sizes[a], sizes[b]
                src, dst = (a, b) if na < nb else (b, a)
                df.loc[(df['ADEP'] == adep) & (df['ADES'] == ades) & (df[label_col] == src), label_col] = dst
                print(f'{adep}-{ades}: C{src} (n={min(na, nb)}) -> C{dst} (n={max(na, nb)})  [L2={min_dist:.1f} nm]')
                changed = True
                break
    for (adep, ades), grp in df.groupby(['ADEP', 'ADES']):
        remap = {old: new for new, old in enumerate(sorted(grp[label_col].unique()))}
        df.loc[(df['ADEP'] == adep) & (df['ADES'] == ades), label_col] = grp[label_col].map(remap)
    return df

df_sample = merge_similar_clusters(df_sample, ACTIVE_FIRS)

print()
print('Post-merge cluster counts:')
print(df_sample.groupby(['ADEP', 'ADES', 'cluster_kmeans']).size().rename('n').reset_index().to_string(index=False))


In [ ]:
# Cell 8
# helper to parse HH:MM duration strings; aggregate per-cluster stats (count, duration, AC type, mean FIR distances)

def _parse_duration(s):
    if isinstance(s, str) and ':' in s:
        h, m = s.split(':')
        return int(h) + int(m) / 60
    return float(s)


def cluster_summary(df, fir_cols, label_col='cluster_kmeans'):
    df = df.copy()
    df['duration_h'] = df['Duration_Hours'].apply(_parse_duration)

    ac_col = next(c for c in df.columns if 'AC Type' in c)

    agg = {
        'n_flights':       ('ECTRL ID', 'count'),
        'mean_duration_h': ('duration_h', 'mean'),
        'most_common_ac':  (ac_col, lambda x: x.mode().iloc[0]),
    }
    agg.update({f'mean_{f}': (f, 'mean') for f in fir_cols})

    return df.groupby(['ADEP', 'ADES', label_col]).agg(**agg).reset_index()

In [ ]:
# Cell 9
# compute the summary table and inspect it

summary = cluster_summary(df_sample, ACTIVE_FIRS)

display(summary[['ADEP', 'ADES', 'cluster_kmeans', 'n_flights', 'mean_duration_h', 'most_common_ac']])
display(summary)

In [ ]:
# Cell 10
# compare KMeans vs DBSCAN cluster counts side by side per O-D pair

for (adep, ades), grp in df_sample.groupby(['ADEP', 'ADES']):
    km = grp['cluster_kmeans'].value_counts().sort_index().to_dict()
    db = grp['cluster_dbscan'].value_counts().sort_index().to_dict()
    print(f"{adep}-{ades}")
    print(f"  KMeans:  {km}")
    print(f"  DBSCAN:  {db}")

In [ ]:
# Cell 11
# show which FIRs define each cluster - reveals whether clusters are genuinely different routes

sigs = make_route_signatures(df_sample, ACTIVE_FIRS)

for (adep, ades), grp in df_sample.groupby(['ADEP', 'ADES']):
    print(f'\n{adep}-{ades}')
    for c in sorted(grp['cluster_kmeans'].unique()):
        mask       = grp['cluster_kmeans'] == c
        modal_firs = sigs.loc[grp[mask].index].columns[
            sigs.loc[grp[mask].index].mean() >= 0.5
        ].tolist()
        print(f'  Cluster {c} (n={mask.sum():>4}):  {modal_firs}')

In [ ]:
# Cell 12
# clustering quality per O-D pair: silhouette, between-cluster centroid separation, within-cluster compactness

from sklearn.metrics.pairwise import pairwise_distances

sigs = make_route_signatures(df_sample, ACTIVE_FIRS)

header = '{:<15}  {:>3}  {:>10}  {:>15}  {:>14}'.format('O-D', 'k', 'silhouette', 'between_hamming', 'within_hamming')
print(header)
for (adep, ades), grp in df_sample.groupby(['ADEP', 'ADES']):
    X      = sigs.loc[grp.index].values.astype(float)
    labels = grp['cluster_kmeans'].values
    k      = len(set(labels))

    if k == 1:
        print(f'{adep}-{ades:<9}  {k:>3}  {"n/a":>10}  {"n/a":>15}  {"n/a":>14}')
        continue

    sil = silhouette_score(X, labels, metric='hamming')

    centroids    = np.array([X[labels == c].mean(axis=0) for c in sorted(set(labels))])
    b_dist       = pairwise_distances(centroids, metric='hamming')
    np.fill_diagonal(b_dist, np.nan)
    mean_between = np.nanmean(b_dist)

    within_vals = [
        pairwise_distances(X[labels == c], metric='hamming').mean()
        for c in sorted(set(labels)) if (labels == c).sum() > 1
    ]
    mean_within = np.mean(within_vals) if within_vals else np.nan

    print(f'{adep}-{ades:<9}  {k:>3}  {sil:>10.4f}  {mean_between:>15.4f}  {mean_within:>14.4f}')


In [ ]:
# Cell 13
# MTOW and cruise fuel flow lookup tables by AC type; map MTOW onto df_sample

MTOW_TONNES = {
    'B77W': 347.4,   # Boeing 777-200LR
    'B772': 297.6,   # Boeing 777-200ER
    'B773': 299.4,   # Boeing 777-300
    'B77L': 347.4,   # Boeing 777F (same MTOW as 200LR)
    'A35K': 280.0,   # Airbus A350-900
    'A359': 280.0,   # Airbus A350-900 (alt code)
    'A21N': 97.0,    # Airbus A321neo
    'A321': 93.5,    # Airbus A321
    'A320': 78.0,    # Airbus A320
    'A20N': 79.0,    # Airbus A320neo
    'A319': 75.5,    # Airbus A319
    'A19N': 75.5,    # Airbus A319neo
    'B738': 79.016,  # Boeing 737-800
    'B737': 65.3,    # Boeing 737-700
    'B739': 85.1,    # Boeing 737-900
    'BCS3': 70.9,    # Airbus A220-300 (CS300)
    'AT43': 16.9,    # ATR 42-300
    'B38M': 82.2,    # Boeing 737 MAX 8
    'A339': 251.0,   # Airbus A330-900neo
    'A333': 242.0,   # Airbus A330-300
    'CRJ9': 38.3,    # Bombardier CRJ900
    'E195': 52.3,    # Embraer E195
    'B764': 204.1,   # Boeing 767-400ER
    'DH8B': 16.5,    # DHC-8-200
    'B789': 254.0,   # Boeing 787-9
    # Regional turboprops
    'AT76': 23.0,   # ATR 72-600
    'AT75': 22.8,   # ATR 72-500
    'AT45': 18.6,   # ATR 42-500
    'DH8D': 29.6,   # Dash 8-400 / Q400
    'DH8A': 15.7,   # Dash 8-100
    # Embraer jets
    'E190': 51.8,   # Embraer 190
    'E170': 37.2,   # Embraer 170
    'E75L': 40.4,   # Embraer E175 (high gross weight)
    'E75S': 38.8,   # Embraer E175 (standard)
    'E295': 61.5,   # Embraer E195-E2
    'E145': 22.0,   # Embraer ERJ-145
    # Bombardier
    'CRJX': 38.3,   # CRJ-900 (most common CRJX variant)
    # Widebodies not yet covered
    'A332': 242.0,  # Airbus A330-200
    'A388': 575.0,  # Airbus A380-800
    'B788': 227.9,  # Boeing 787-8
    'B78X': 254.0,  # Boeing 787-10
    'B763': 186.9,  # Boeing 767-300
    'B752': 115.7,  # Boeing 757-200
    'B734':  68.0,  # Boeing 737-400
    'SU95':  49.5,  # Sukhoi Superjet 100
}

# Cruise fuel flow (kg/h) - ICAO FEAT / EcoScope block-hour averages at typical cruise
FUEL_KGH = {
    'B77W': 7800,   'B772': 6600,   'B773': 7000,   'B77L': 7800,
    'A35K': 5800,   'A359': 5800,   'A339': 5400,   'A333': 6200,
    'B764': 5500,   'B789': 5500,
    'A21N': 2500,   'A321': 2800,   'A320': 2500,   'A20N': 2300,
    'A319': 2200,   'A19N': 2000,
    'B38M': 2100,   'B738': 2400,   'B737': 2200,   'B739': 2600,
    'BCS3': 1900,   'CRJ9': 1400,   'E195': 1800,
    'AT43':  600,   'DH8B':  500,
    'AT76':  900,   'AT75':  850,   'AT45':  650,
    'DH8D': 1150,   'DH8A':  600,
    'E190': 2200,   'E170': 1700,   'E75L': 1800,   'E75S': 1800,
    'E295': 2600,   'E145': 1350,   'CRJX': 2000,
    'A332': 6000,   'A388': 12000,
    'B788': 5200,   'B78X': 6500,   'B763': 5500,   'B752': 4000,
    'B734': 3100,   'SU95': 2300,
}

JET_A_EUR_PER_KG = 0.81   # EIA US Gulf Coast FOB avg Sep 2023 (~$2.56/gal) + CIF NWE premium, EUR/USD 1.074

ac_col              = next(c for c in df_sample.columns if 'AC Type' in c)
df_sample['mtow_t'] = df_sample[ac_col].map(MTOW_TONNES)

missing = df_sample[df_sample['mtow_t'].isna()][ac_col].unique()
print(f"Missing MTOW: {df_sample['mtow_t'].isna().sum()} flights - types: {missing}")
print(df_sample[[ac_col, 'mtow_t']].drop_duplicates().sort_values(ac_col))

In [ ]:
# Cell 14
# Sep 2023 Eurocontrol en-route unit rates (EUR per service unit)
# Source: Eurocontrol CRCO Information Circular 2023/01 (effective 1 Jan 2023)
#         + September 2023 monthly adjusted rates for CHF/RSD-denominated states
EUROCONTROL_RATES = {
    # --- Atlantic / North Atlantic (EGLL-KJFK) ---
    'BGGLFIR': 61.04,   # Denmark / Naviair (Greenland)
    'EGGXFIR': 87.88,   # UK / NATS (Shanwick Oceanic)
    'EGTTFIR': 87.88,   # UK / NATS (London FIR)
    'EGTTUIR': 87.88,   # UK / NATS (London UIR)
    'EISNUIR': 26.46,   # Ireland / IAA (Shannon UIR)
    'ENORFIR': 47.66,   # Norway / Avinor

    # --- Iberian / Iberian-Atlantic (LEBL-LEPA, LPPT-EDDB) ---
    'LECBFIR': 54.71,   # Spain / ENAIRE (Continental)
    'LPPOFIR': 10.03,   # Portugal / NAV (Santa Maria Oceanic)
    'LPPCFIR': 47.39,   # Portugal / NAV (Lisbon FIR mainland)     - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)

    # --- Continental Europe (LPPT-EDDB) ---
    'LFBBFIR': 73.69,   # France / DSNA (Bordeaux FIR)             - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)
    'LFFFFIR': 73.69,   # France / DSNA (Paris FIR)                - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)
    'LFEEFIR': 73.69,   # France / DSNA (Reims FIR)                - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)
    'EDGGFIR': 73.04,   # Germany / DFS (Langen FIR)               - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)
    'EDMMFIR': 73.04,   # Germany / DFS (Munich FIR)               - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)
    'EDWWFIR': 73.04,   # Germany / DFS (Bremen FIR)               - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)

    # --- LGAV-EIDW validation pair additions ---
    'LGGGFIR': 25.54,   # Greece / HCAA (Athens FIR lower)         - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)
    'LGGGUIR': 25.54,   # Greece / HCAA (Athens UIR upper)         - same rate as lower
    'LIRRUIR': 72.37,   # Italy / ENAV (Roma UIR)                  - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected); same as lower FIR
    'LIMMUIR': 72.37,   # Italy / ENAV (Milano UIR)                - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected); same as lower FIR
    'LFFFUIR': 73.69,   # France / DSNA (Paris UIR)                - same rate as all French FIRs
    'LJLAFIR': 65.32,   # Slovenia / Sloveniacontrol (Ljubljana)   - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)
    'LOVVFIR': 66.91,   # Austria / Austro Control (Vienna)        - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)
    'LDZOFIR': 45.83,   # Croatia / Croatia Control (Zagreb)       - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)
    'LSASUIR': 120.92,  # Switzerland / skyguide (Geneva UIR)      - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)
    'EBURUIR': 113.21,  # Belgium-Luxembourg / skeyes (Brussels UIR) - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (corrected)
    'EDUUUIR': 73.04,   # Germany / DFS (upper UIR)                - same rate as lower FIRs
    'LAAAFIR': 55.71,   # Albania / Albcontrol                    - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (verified)
    'LYBAUIR': 39.50,   # Serbia-Montenegro-KFOR / SMATSA          - Eurocontrol CRCO Sep 2023 monthly adjusted global unit rate (verified)

    # --- NAT corridor additions (present in actual trajectory data) ---
    'EGPXUIR': 87.88,   # UK / NATS (Scottish UIR / Prestwick Oceanic)     - same rate as EGGXFIR/EGTTUIR
    'BIRDFIR': 66.80,   # Iceland / Isavia (Reykjavik FIR)                 - est. from Eurocontrol CRCO IC 2023/01; verify
}

# Nav Canada 2024 rates used as proxy for 2023 (<3% annual change)
# Source: Nav Canada Customer Guide to Charges, effective January 1, 2024
NAV_CANADA_R       = 0.03402   # CAD / (km x sqrt(tonne)) - domestic enroute formula
NAV_CANADA_OCEANIC = 210.19    # CAD flat fee - Gander Oceanic (NAT $183.90 + Int'l Comm $26.29)
CAD_EUR            = 0.695     # Aug 2023 average exchange rate (Reuters closing cross rate)

# CZQXFIR (Gander Oceanic): flat fee - enroute charge excludes this FIR per Nav Canada rules
# CZQMFIR (Moncton) + CZULFIR (Montreal): distance-based enroute formula
# KZBWFIR, KZBWUIR, KZNYFIR, KZWYFIR: zero - EGLL-KJFK lands in US, no FAA overflight fee

In [ ]:
# Cell 15
# per-flight cost split: flight_atc_eur (Eurocontrol + Nav Canada) and flight_fuel_eur; total = sum

def flight_atc_eur(row):
    mtow = row['mtow_t']
    if pd.isna(mtow):
        return np.nan

    # Sum FIR + UIR distances per ANSP base code: same charging zone, different altitude bands
    base_dists = {}
    base_rates = {}
    for fir, rate in EUROCONTROL_RATES.items():
        d = row.get(fir, 0) or 0
        base = fir[:-3]
        base_dists[base] = base_dists.get(base, 0) + d
        if base not in base_rates:
            base_rates[base] = rate

    total = 0.0
    for base, d_nm in base_dists.items():
        if d_nm <= 0:
            continue
        dist_km = d_nm * 1.852
        su = (dist_km / 100) * np.sqrt(mtow / 50)
        total += su * base_rates[base]

    for fir in ('CZQMFIR', 'CZULFIR'):
        dist_nm = row.get(fir, 0)
        if dist_nm > 0:
            dist_km = dist_nm * 1.852
            total  += NAV_CANADA_R * np.sqrt(mtow) * dist_km * CAD_EUR
    if row.get('CZQXFIR', 0) > 0:
        total += NAV_CANADA_OCEANIC * CAD_EUR
    return round(total, 2)


def flight_fuel_eur(row):
    fuel_kgh = FUEL_KGH.get(row[ac_col])
    if fuel_kgh is None or pd.isna(row['mtow_t']):
        return np.nan
    return round(fuel_kgh * _parse_duration(row['Duration_Hours']) * JET_A_EUR_PER_KG, 2)


df_sample['atc_eur']  = df_sample.apply(flight_atc_eur, axis=1)
df_sample['fuel_eur'] = df_sample.apply(flight_fuel_eur, axis=1)
df_sample['cost_eur'] = (df_sample['atc_eur'] + df_sample['fuel_eur']).round(2)

cost_by_cluster = (
    df_sample.groupby(['ADEP', 'ADES', 'cluster_kmeans'])[['atc_eur', 'fuel_eur', 'cost_eur']]
    .agg(['mean', 'std'])
    .round(2)
)
print(cost_by_cluster)

cost_agg = (
    df_sample.groupby(['ADEP', 'ADES', 'cluster_kmeans'])['cost_eur']
    .agg(mean_cost_eur='mean', std_cost_eur='std')
    .round(2)
    .reset_index()
)
summary = summary.drop(columns=['mean_cost_eur', 'std_cost_eur'], errors='ignore')
summary = summary.merge(cost_agg, on=['ADEP', 'ADES', 'cluster_kmeans'], how='left')

In [ ]:
# Cell 16
# ML features: encode categoricals, count active FIRs, compute total distance; set up atc and fuel targets

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df_ml = df_sample.copy()
df_ml['duration_h']     = df_ml['Duration_Hours'].apply(_parse_duration)
df_ml['n_firs_crossed'] = (df_ml[ACTIVE_FIRS] > 0).sum(axis=1)
df_ml['total_dist_nm']  = df_ml[ACTIVE_FIRS].sum(axis=1)

ac_col_name = next(c for c in df_ml.columns if 'AC Type' in c)
le_ac = LabelEncoder()
le_od = LabelEncoder()
df_ml['ac_enc'] = le_ac.fit_transform(df_ml[ac_col_name].fillna('UNKNOWN'))
df_ml['od_enc'] = le_od.fit_transform(df_ml['ADEP'] + '-' + df_ml['ADES'])

df_ml = df_ml.dropna(subset=['atc_eur', 'fuel_eur', 'mtow_t'])

FEATURES = ['duration_h', 'mtow_t', 'n_firs_crossed', 'total_dist_nm', 'ac_enc', 'od_enc'] + ACTIVE_FIRS

X      = df_ml[FEATURES].fillna(0).values
y_atc  = df_ml['atc_eur'].values
y_fuel = df_ml['fuel_eur'].values
y      = df_ml['cost_eur'].values

print(f'ML dataset: {len(X)} rows, {len(FEATURES)} features')
print(f'ATC  target: €{y_atc.min():.0f} – €{y_atc.max():.0f}  (mean €{y_atc.mean():.0f})')
print(f'Fuel target: €{y_fuel.min():.0f} – €{y_fuel.max():.0f}  (mean €{y_fuel.mean():.0f})')
print(f'Total:       €{y.min():.0f} – €{y.max():.0f}  (mean €{y.mean():.0f})')

In [ ]:
# Cell 17
# train Ridge + RF for atc_eur and fuel_eur separately; at predict time sum both for total cost

X_train, X_test, y_atc_train, y_atc_test, y_fuel_train, y_fuel_test = train_test_split(
    X, y_atc, y_fuel, test_size=0.2, random_state=158
)

scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)
X_all_sc = scaler.transform(X)

ridge_atc = Ridge(alpha=100)
ridge_atc.fit(X_tr_sc, y_atc_train)
y_pred_ridge_atc = ridge_atc.predict(X_te_sc)
cv_ridge_atc     = cross_val_score(Ridge(alpha=100), X_all_sc, y_atc, cv=5, scoring='r2')

rf_atc = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=158, n_jobs=-1)
rf_atc.fit(X_train, y_atc_train)
y_pred_rf_atc = rf_atc.predict(X_test)
cv_rf_atc     = cross_val_score(
    RandomForestRegressor(n_estimators=300, max_depth=8, random_state=158, n_jobs=-1),
    X, y_atc, cv=5, scoring='r2'
)

ridge_fuel = Ridge(alpha=100)
ridge_fuel.fit(X_tr_sc, y_fuel_train)
y_pred_ridge_fuel = ridge_fuel.predict(X_te_sc)
cv_ridge_fuel     = cross_val_score(Ridge(alpha=100), X_all_sc, y_fuel, cv=5, scoring='r2')

rf_fuel = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=158, n_jobs=-1)
rf_fuel.fit(X_train, y_fuel_train)
y_pred_rf_fuel = rf_fuel.predict(X_test)
cv_rf_fuel     = cross_val_score(
    RandomForestRegressor(n_estimators=300, max_depth=8, random_state=158, n_jobs=-1),
    X, y_fuel, cv=5, scoring='r2'
)

print(f'{"Model":<24}  {"Target":<6}  {"R²":>7}  {"MAE":>9}  {"RMSE":>9}  {"CV R²":>14}')
for name, target, y_pred, y_t, cv in [
    ('Ridge',         'ATC',  y_pred_ridge_atc,  y_atc_test,  cv_ridge_atc),
    ('Random Forest', 'ATC',  y_pred_rf_atc,     y_atc_test,  cv_rf_atc),
    ('Ridge',         'Fuel', y_pred_ridge_fuel, y_fuel_test, cv_ridge_fuel),
    ('Random Forest', 'Fuel', y_pred_rf_fuel,    y_fuel_test, cv_rf_fuel),
]:
    print(f'{name:<24}  {target:<6}  {r2_score(y_t, y_pred):>7.4f}  '
          f'€{mean_absolute_error(y_t, y_pred):>7.1f}  '
          f'€{np.sqrt(mean_squared_error(y_t, y_pred)):>7.1f}  '
          f'{cv.mean():>6.4f} ± {cv.std():.4f}')

for label, model in [('ATC', rf_atc), ('Fuel', rf_fuel)]:
    fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False).head(10)
    print(f'\nTop 10 features (RF {label}):')
    print(fi.round(4))

In [ ]:
# Cell 18
# build representative feature vectors per cluster; predicted_cost = rf_atc + rf_fuel

X_cluster_rows = []
for _, row in summary.iterrows():
    adep, ades, cluster = row['ADEP'], row['ADES'], row['cluster_kmeans']
    grp = df_ml[
        (df_ml['ADEP'] == adep) &
        (df_ml['ADES'] == ades) &
        (df_ml['cluster_kmeans'] == cluster)
    ]
    ac_type = row['most_common_ac']
    feats = [
        row['mean_duration_h'],
        MTOW_TONNES.get(ac_type, grp['mtow_t'].mean() if not grp.empty else 0),
        int((grp[ACTIVE_FIRS] > 0).mean().gt(0).sum()) if not grp.empty else 0,
        float(grp[ACTIVE_FIRS].mean().sum()) if not grp.empty else 0,
        le_ac.transform([ac_type])[0] if ac_type in le_ac.classes_ else 0,
        le_od.transform([f'{adep}-{ades}'])[0],
    ] + [row.get(f'mean_{fir}', 0) for fir in ACTIVE_FIRS]
    X_cluster_rows.append(feats)

X_out = np.nan_to_num(np.array(X_cluster_rows, dtype=float), nan=0.0)
summary['predicted_atc_eur']  = rf_atc.predict(X_out).round(2)
summary['predicted_fuel_eur'] = rf_fuel.predict(X_out).round(2)
summary['predicted_cost_eur'] = (summary['predicted_atc_eur'] + summary['predicted_fuel_eur']).round(2)

OUTPUT_COLS = ['ADEP', 'ADES', 'cluster_kmeans', 'n_flights', 'mean_duration_h',
               'most_common_ac', 'mean_cost_eur', 'std_cost_eur',
               'predicted_atc_eur', 'predicted_fuel_eur', 'predicted_cost_eur']
display(summary[OUTPUT_COLS])

out_path = '/drive/MyDrive/flight-project/route_alternatives.csv'
summary[OUTPUT_COLS].to_csv(out_path, index=False)
print(f'Saved to {out_path}')

In [ ]:
# Cell 19
# query function: formula-based cost prediction per cluster, ranked by cost or duration

def predict_route_options(ac_type, adep, ades, sort_by='cost_eur'):
    od_rows = summary[(summary['ADEP'] == adep) & (summary['ADES'] == ades)]
    if od_rows.empty:
        raise ValueError(f'No clusters found for {adep}-{ades}')

    mtow = MTOW_TONNES.get(ac_type)
    if mtow is None:
        raise ValueError(f"AC type '{ac_type}' not in MTOW_TONNES - add it to the dict")

    fuel_kgh = FUEL_KGH.get(ac_type)
    if fuel_kgh is None:
        raise ValueError(f"AC type '{ac_type}' not in FUEL_KGH - add it to the dict")

    results = []
    for _, row in od_rows.iterrows():
        duration = row['mean_duration_h']

        rep_row = {fir: row.get(f'mean_{fir}', 0)
                   for fir in list(EUROCONTROL_RATES) + ['CZQMFIR', 'CZULFIR', 'CZQXFIR']}
        rep_row['mtow_t'] = mtow

        pred_atc  = round(flight_atc_eur(rep_row), 2)
        pred_fuel = round(fuel_kgh * duration * JET_A_EUR_PER_KG, 2)

        results.append({
            'cluster':              row['cluster_kmeans'],
            'n_historical_flights': int(row['n_flights']),
            'mean_duration_h':      round(duration, 3),
            'predicted_atc_eur':    pred_atc,
            'predicted_fuel_eur':   pred_fuel,
            'predicted_cost_eur':   round(pred_atc + pred_fuel, 2),
        })

    sort_col = 'predicted_cost_eur' if sort_by == 'cost_eur' else 'mean_duration_h'
    df_out = (
        pd.DataFrame(results)
        .sort_values(sort_col)
        .reset_index(drop=True)
    )
    df_out.index      = df_out.index + 1
    df_out.index.name = 'rank'
    return df_out

In [ ]:
# Cell 20
# example queries - swap in any AC type, ADEP, ADES, and sort_by metric

print('A350-900 | EGLL-KJFK | ranked by cost')
display(predict_route_options('A35K', 'EGLL', 'KJFK', sort_by='cost_eur'))

print('\nB737-800 | LEBL-LEPA | ranked by duration')
display(predict_route_options('B738', 'LEBL', 'LEPA', sort_by='duration_h'))

print('\nA320 | LPPT-EDDB | ranked by cost')
display(predict_route_options('A320', 'LPPT', 'EDDB', sort_by='cost_eur'))

In [ ]:
# Cell 21
# interactive widget UI -- routes drawn from full_summary (7,345 pairs) when available

import ipywidgets as widgets
from IPython.display import display, clear_output

ac_options = sorted(MTOW_TONNES.keys())

try:
    od_set = (
        full_summary[['ADEP', 'ADES']].drop_duplicates()
        .apply(lambda r: f"{r['ADEP']}-{r['ADES']}", axis=1)
        .sort_values().tolist()
    )
    _src_label = f'full dataset ({len(od_set):,} pairs)'
except NameError:
    od_set = (
        df_sample[['ADEP', 'ADES']].drop_duplicates()
        .apply(lambda r: f"{r['ADEP']}-{r['ADES']}", axis=1)
        .tolist()
    )
    _src_label = 'training sample (3 pairs) -- run cells 29-33 for full coverage'

dd_ac   = widgets.Dropdown(options=ac_options, description='AC Type:',
                            layout=widgets.Layout(width='220px'))
txt_od  = widgets.Text(description='Route:', placeholder='EGLL-KJFK',
                        layout=widgets.Layout(width='220px'))
dd_sort = widgets.Dropdown(options=['cost_eur', 'duration_h'], description='Sort by:',
                            layout=widgets.Layout(width='220px'))
btn     = widgets.Button(description='Predict', button_style='primary')
lbl     = widgets.Label(value=f'Source: {_src_label}')
out     = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        val = txt_od.value.strip().upper()
        if not val or '-' not in val:
            print('Enter a route as ADEP-ADES (e.g. EGLL-KJFK)')
            return
        parts = val.split('-')
        if len(parts) != 2 or not all(len(p) == 4 for p in parts):
            print('Both airport codes must be 4-letter ICAO (e.g. EGLL-KJFK)')
            return
        adep, ades = parts
        if val not in od_set:
            print(f'{val} not in dataset -- pair has fewer than 30 flights or airport code is wrong')
            return
        try:
            display(predict_route_options(dd_ac.value, adep, ades, sort_by=dd_sort.value))
        except ValueError as e:
            print(f'Error: {e}')

btn.on_click(on_click)
display(widgets.VBox([lbl, dd_ac, txt_od, dd_sort, btn, out]))


In [ ]:
# Cell 22
# EGLL-LGAV: load held-out validation pair (saved during rebuild in cell 1)

if not os.path.exists(VAL_CACHE):
    raise RuntimeError('VAL_CACHE not found - run cell 1 with force_rebuild=True to build it')

df_val = pd.read_csv(VAL_CACHE)

_val_fir_cands  = [c for c in df_val.columns if c.endswith(('FIR', 'UIR'))]
VAL_ACTIVE_FIRS = [f for f in _val_fir_cands if (df_val[f] > 0).mean() >= 0.25]

print(f'EGLL-LGAV: {len(df_val)} flights')
print(f'Active FIRs ({len(VAL_ACTIVE_FIRS)}): {VAL_ACTIVE_FIRS}')

In [ ]:
# Cell 23
# EGLL-LGAV: cluster → atc_eur + fuel_eur (formula path, no RF) → MAE / R²

_ac_col_val  = next(c for c in df_val.columns if 'AC Type' in c)
ac_col       = _ac_col_val

df_val['mtow_t']     = df_val[_ac_col_val].map(MTOW_TONNES)
df_val['duration_h'] = df_val['Duration_Hours'].apply(_parse_duration)
df_val['atc_eur']    = df_val.apply(flight_atc_eur, axis=1)
df_val['fuel_eur']   = df_val.apply(flight_fuel_eur, axis=1)
df_val['cost_eur']   = (df_val['atc_eur'] + df_val['fuel_eur']).round(2)

print(f'Missing MTOW: {df_val["mtow_t"].isna().sum()} | Missing cost: {df_val["cost_eur"].isna().sum()}')
print(f'Actual cost range: €{df_val["cost_eur"].min():.0f} – €{df_val["cost_eur"].max():.0f}  '
      f'(mean €{df_val["cost_eur"].mean():.0f})')
print('Note: Albania (LAAAFIR) and Serbia (LYBAUIR) rates verified against Eurocontrol CRCO Sep 2023 monthly adjusted unit rates.\n')

_, df_val['cluster_kmeans'] = cluster_od_3layer(df_val, VAL_ACTIVE_FIRS)
# without this, validation skips the same size-guarantee fix applied to df_sample
df_val = merge_similar_clusters(df_val, VAL_ACTIVE_FIRS)

val_summary = cluster_summary(df_val, VAL_ACTIVE_FIRS)
val_cost_agg = (
    df_val.dropna(subset=['cost_eur'])
    .groupby(['ADEP', 'ADES', 'cluster_kmeans'])['cost_eur']
    .agg(mean_cost_eur='mean', std_cost_eur='std').round(2).reset_index()
)
val_summary = val_summary.merge(val_cost_agg, on=['ADEP', 'ADES', 'cluster_kmeans'], how='left')
display(val_summary[['cluster_kmeans', 'n_flights', 'mean_duration_h', 'most_common_ac',
                      'mean_cost_eur', 'std_cost_eur']])

_all_fir_cols = [c for c in df_val.columns if c.endswith(('FIR', 'UIR'))]

cluster_pred_map = {}
print('\nFormula predictions per cluster (most_common_ac, mean FIR distances):')
for _, crow in val_summary.iterrows():
    ac_type  = crow['most_common_ac']
    cluster  = crow['cluster_kmeans']
    mtow     = MTOW_TONNES.get(ac_type)
    fuel_kgh = FUEL_KGH.get(ac_type)
    if mtow is None or fuel_kgh is None:
        print(f'  Cluster {cluster}: {ac_type} missing from lookup tables - skipped')
        continue

    grp      = df_val[df_val['cluster_kmeans'] == cluster]
    duration = crow['mean_duration_h']

    rep_row = grp[_all_fir_cols].fillna(0).mean().to_dict()
    rep_row['mtow_t']         = mtow
    rep_row['Duration_Hours'] = duration
    rep_row[_ac_col_val]      = ac_type

    pred_atc  = round(flight_atc_eur(rep_row), 2)
    pred_fuel = round(fuel_kgh * duration * JET_A_EUR_PER_KG, 2)
    predicted = round(pred_atc + pred_fuel, 2)
    cluster_pred_map[cluster] = predicted
    print(f'  Cluster {cluster} ({ac_type}, n={int(crow["n_flights"])}):  '
          f'atc €{pred_atc:,.0f} + fuel €{pred_fuel:,.0f} = predicted €{predicted:,.0f}  '
          f'actual mean €{crow["mean_cost_eur"]:,.0f}')

df_eval = df_val.dropna(subset=['cost_eur', 'mtow_t']).copy()
df_eval['predicted_cost_eur'] = df_eval['cluster_kmeans'].map(cluster_pred_map)
df_eval = df_eval.dropna(subset=['predicted_cost_eur'])

y_act  = df_eval['cost_eur'].values
y_pred = df_eval['predicted_cost_eur'].values

mae = mean_absolute_error(y_act, y_pred)
r2  = r2_score(y_act, y_pred) if len(set(y_pred)) > 1 else float('nan')

print(f'\nEGLL-LGAV out-of-sample validation (n={len(df_eval)} flights)')
print(f'MAE:  €{mae:,.0f}')
if np.isnan(r2):
    print('R²:   n/a - single cluster, no between-cluster variance')
else:
    print(f'R²:   {r2:.4f}')

print(f'\nmean actual:    €{y_act.mean():,.0f}')
print(f'mean predicted: €{y_pred.mean():,.0f}')

In [ ]:
# Cell 24
# FIR usage heatmap per O-D pair: which FIRs define each cluster

import plotly.graph_objects as go

def plot_fir_heatmap(df, fir_cols, label_col='cluster_kmeans'):
    for (adep, ades), grp in df.groupby(['ADEP', 'ADES']):
        clusters  = sorted(grp[label_col].unique())
        pair_firs = [f for f in fir_cols if (grp[f] > 0).mean() >= 0.10]
        if not pair_firs:
            continue

        n_list   = [grp[grp[label_col] == c].shape[0] for c in clusters]
        z        = [[round(grp[grp[label_col] == c][f].mean(), 1) for f in pair_firs] for c in clusters]
        y_labels = [f'C{c}  (n={n_list[i]})' for i, c in enumerate(clusters)]
        x_labels = [f.replace('FIR', '').replace('UIR', '') for f in pair_firs]

        fig = go.Figure(go.Heatmap(
            z=z, x=x_labels, y=y_labels,
            colorscale=[[0, 'green'], [0.5, 'yellow'], [1, 'red']], colorbar_title='mean nm'
        ))
        fig.update_layout(
            title=f'{adep}-{ades}: FIR usage by cluster (mean distance, nm)',
            xaxis_tickangle=-45,
            height=max(300, len(clusters) * 55 + 150),
            margin=dict(l=130, b=130)
        )
        fig.show()

plot_fir_heatmap(df_sample, ACTIVE_FIRS)

_val_fir_cols = [c for c in df_val.columns if c.endswith(('FIR', 'UIR'))]
plot_fir_heatmap(df_val, _val_fir_cols)


In [ ]:
# Cell 25
# cost breakdown by cluster: ATC vs fuel, with total and n annotated

def plot_cost_bars(df, label_col='cluster_kmeans'):
    for (adep, ades), grp in df.groupby(['ADEP', 'ADES']):
        grp      = grp.dropna(subset=['atc_eur', 'fuel_eur'])
        clusters = sorted(grp[label_col].unique())
        x        = [f'C{c}' for c in clusters]
        atc_m    = [round(grp[grp[label_col] == c]['atc_eur'].mean()) for c in clusters]
        fuel_m   = [round(grp[grp[label_col] == c]['fuel_eur'].mean()) for c in clusters]
        n_list   = [grp[grp[label_col] == c].shape[0] for c in clusters]

        fig = go.Figure(data=[
            go.Bar(name='ATC',  x=x, y=atc_m,  marker_color='#4C78A8'),
            go.Bar(name='Fuel', x=x, y=fuel_m, marker_color='#F58518'),
        ])
        for xi, atc, fuel, n in zip(x, atc_m, fuel_m, n_list):
            fig.add_annotation(
                x=xi, y=atc + fuel,
                text=f'€{atc + fuel:,}<br>n={n}',
                showarrow=False, yanchor='bottom', font_size=11
            )
        fig.update_layout(
            barmode='stack',
            title=f'{adep}-{ades}: mean cost per cluster',
            yaxis_title='EUR',
            height=460
        )
        fig.show()

plot_cost_bars(df_sample)
plot_cost_bars(df_val)


In [ ]:
# Cell 26
# PCA scatter: reduce binary FIR signatures to 2D, coloured by cluster - shows cluster separation

from sklearn.decomposition import PCA
import plotly.express as px

def plot_cluster_pca(df, fir_cols, label_col='cluster_kmeans'):
    sigs = (df[fir_cols] > 0).astype(int)

    for (adep, ades), grp in df.groupby(['ADEP', 'ADES']):
        k = grp[label_col].nunique()
        if k < 2:
            print(f'{adep}-{ades}: single cluster - skipped')
            continue

        X      = sigs.loc[grp.index].values
        coords = PCA(n_components=2, random_state=158).fit_transform(X)

        plot_df = pd.DataFrame({
            'PC1':     coords[:, 0],
            'PC2':     coords[:, 1],
            'cluster': grp[label_col].astype(str).values,
        })

        fig = px.scatter(
            plot_df, x='PC1', y='PC2', color='cluster',
            title=f'{adep}-{ades}: cluster separation (PCA of binary FIR signatures)',
            labels={'cluster': 'Cluster'},
        )
        fig.update_traces(marker_size=6, marker_opacity=0.7)
        fig.update_layout(height=500)
        fig.show()

plot_cluster_pca(df_sample, ACTIVE_FIRS)

_val_fir_cols = [c for c in df_val.columns if c.endswith(('FIR', 'UIR'))]
plot_cluster_pca(df_val, _val_fir_cols)


In [ ]:
# Cell 26
# cost vs duration scatter: one point per cluster, sized by n_flights - direct view of route alternatives

def plot_route_alternatives(df_sum, title_suffix=''):
    for (adep, ades), grp in df_sum.groupby(['ADEP', 'ADES']):
        if grp['mean_cost_eur'].isna().all():
            continue

        fig = px.scatter(
            grp,
            x='mean_duration_h',
            y='mean_cost_eur',
            size='n_flights',
            color=grp['cluster_kmeans'].astype(str),
            text=grp['cluster_kmeans'].astype(str),
            title=f'{adep}-{ades}: route alternatives{title_suffix}',
            labels={
                'mean_duration_h': 'Mean duration (h)',
                'mean_cost_eur':   'Mean cost (€)',
                'color':           'Cluster',
            },
            size_max=40,
        )
        fig.update_traces(textposition='top center', marker_opacity=0.85)
        fig.update_layout(height=520, showlegend=False)
        fig.show()

plot_route_alternatives(summary)
plot_route_alternatives(val_summary, title_suffix=' - EGLL-LGAV validation')


In [ ]:
# Cell 27 - full-dataset O-D pair profiling
# Counts flights per O-D pair across the entire Sep 2023 scheduled-flights dataset.
# Uses only the Flights parquet (4 columns) - avoids loading the 305-column FIR parquet.
# 99.92% of scheduled flights survive the inner join, so counts are effectively identical.

FULL_OD_CACHE = '/drive/MyDrive/flight-project/od_counts_full.csv'

if os.path.exists(FULL_OD_CACHE):
    od_counts = pd.read_csv(FULL_OD_CACHE)
    print(f'Loaded from cache: {len(od_counts):,} O-D pairs, {od_counts["n_flights"].sum():,} flights')
else:
    flights_full = pd.read_parquet(
        RAW_FLIGHTS,
        columns=['ECTRL ID', 'ADEP', 'ADES', 'ICAO Flight Type']
    )
    flights_sched = flights_full[flights_full['ICAO Flight Type'] == 'S']
    od_counts = (
        flights_sched.groupby(['ADEP', 'ADES'])
        .size()
        .reset_index(name='n_flights')
        .sort_values('n_flights', ascending=False)
        .reset_index(drop=True)
    )
    del flights_full, flights_sched
    od_counts.to_csv(FULL_OD_CACHE, index=False)
    print(f'Saved: {len(od_counts):,} O-D pairs, {od_counts["n_flights"].sum():,} flights')

print(f'\nTop 10 O-D pairs by flight count:')
print(od_counts.head(10).to_string(index=False))
print(f'\nDistribution summary:')
print(od_counts['n_flights'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(1).to_string())

In [ ]:
# Cell 28 - threshold sweep + CDF visualisation
# MIN_CLUSTER_SIZE=15; worst-case path (k2=2, k3=2 per L2 group) requires 4×15=60 flights.
# Practical floor is ~50–100; distribution elbow determines the right cutoff.

import plotly.graph_objects as go
from plotly.subplots import make_subplots

THRESHOLDS    = [15, 30, 50, 75, 100, 150, 200, 300, 500, 1000]
total_flights = od_counts['n_flights'].sum()
total_pairs   = len(od_counts)

print(f'{"Threshold":>10}  {"Pairs":>8}  {"% pairs":>8}  {"Flights":>12}  {"% flights":>10}')
print('-' * 58)
for t in THRESHOLDS:
    mask = od_counts['n_flights'] >= t
    n_p  = mask.sum()
    n_f  = od_counts.loc[mask, 'n_flights'].sum()
    print(f'{t:>10,}  {n_p:>8,}  {n_p / total_pairs * 100:>7.1f}%  {n_f:>12,}  {n_f / total_flights * 100:>9.1f}%')

# CDF: fraction of pairs that survive each threshold
x_cdf = np.sort(od_counts['n_flights'].values)
y_cdf = 1 - np.arange(1, len(x_cdf) + 1) / len(x_cdf)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Flights per O-D pair (log scale)',
                                    'Fraction of pairs above threshold'])

fig.add_trace(go.Histogram(x=od_counts['n_flights'], nbinsx=100,
                            marker_color='#4C78A8', name='pairs'),
              row=1, col=1)

fig.add_trace(go.Scatter(x=x_cdf, y=y_cdf, mode='lines',
                          line=dict(color='#F58518', width=2)),
              row=1, col=2)

fig.update_xaxes(type='log', title_text='flights per O-D pair', row=1, col=1)
fig.update_xaxes(type='log', title_text='threshold N (min flights per pair)', row=1, col=2)
fig.update_yaxes(title_text='number of O-D pairs', row=1, col=1)
fig.update_yaxes(title_text='fraction of pairs surviving', row=1, col=2)
fig.update_layout(height=430, showlegend=False,
                  title_text='Sep 2023 - O-D pair threshold analysis')
fig.show()

In [ ]:
# Cell 29 - load / build full qualifying dataset (all pairs with ≥30 flights)

MIN_FLIGHTS_PER_OD = 30
FULL_DATA_CACHE    = '/drive/MyDrive/flight-project/df_full.parquet'

if os.path.exists(FULL_DATA_CACHE):
    df_full = pd.read_parquet(FULL_DATA_CACHE)
    print(f'Loaded from cache: {df_full.shape}')
else:
    print('Building from raw parquets (takes ~3–5 min)...')
    fir_full     = pd.read_parquet(RAW_FIR)
    flights_full = pd.read_parquet(RAW_FLIGHTS)
    flights_full = flights_full[flights_full['ICAO Flight Type'] == 'S']
    flights_full.drop(
        columns=[c for c in ['STATFOR Market Segment'] if c in flights_full.columns],
        inplace=True
    )
    df = fir_full.merge(flights_full, on='ECTRL ID', how='inner', suffixes=('', '_drop'))
    del fir_full, flights_full
    df.drop(columns=[c for c in df.columns if c.endswith('_drop')], inplace=True)

    pair_counts = df.groupby(['ADEP', 'ADES']).size()
    qualifying  = pair_counts[pair_counts >= MIN_FLIGHTS_PER_OD].index
    od_idx      = pd.MultiIndex.from_frame(df[['ADEP', 'ADES']])
    df_full     = df[od_idx.isin(qualifying)].reset_index(drop=True)
    del df

    df_full.to_parquet(FULL_DATA_CACHE, index=False)
    print(f'Saved to Drive: {df_full.shape}')

# uses FIR/UIR suffix, not dtype==float64 -- same leak as ACTIVE_FIRS (cell 2) had,
# the dtype filter also matched ADEP/ADES Latitude/Longitude and Requested FL
_fir_all_cols = [c for c in df_full.columns if c.endswith(('FIR', 'UIR'))]

n_pairs_full = df_full.groupby(['ADEP', 'ADES']).ngroups
print(f'\nQualifying pairs (≥{MIN_FLIGHTS_PER_OD} flights): {n_pairs_full:,}')
print(f'Total flights in df_full: {len(df_full):,}')
print(f'FIR feature columns: {len(_fir_all_cols)}')

In [ ]:
# Cell 30 - 3-layer clustering on all qualifying pairs; active FIRs computed per pair
# streams groupby instead of materialising every sub-frame in a list upfront, and runs
# gc.collect() periodically -- the previous version OOM'd at 6500/7345 pairs on Colab's
# free-tier RAM, most likely fragmentation from ~15k KMeans fits (n_init=10, twice per pair)
# plus holding all 7,345 group copies in memory simultaneously via list(df.groupby(...))

FULL_LABELS_CACHE = '/drive/MyDrive/flight-project/cluster_labels_full.csv'
FULL_DIAG_CACHE   = '/drive/MyDrive/flight-project/clust_diagnostic_full.csv'


def cluster_full_dataset(df, all_fir_cols):
    from sklearn.preprocessing import StandardScaler
    labels_final = pd.Series(-1, index=df.index, dtype=int)
    diagnostics  = []
    n            = df.groupby(['ADEP', 'ADES']).ngroups

    for i, ((adep, ades), od_grp) in enumerate(df.groupby(['ADEP', 'ADES'])):
        if (i + 1) % 500 == 0 or i == n - 1:
            print(f'  {i + 1}/{n} pairs processed', flush=True)
            gc.collect()

        pair_firs = [f for f in all_fir_cols if (od_grp[f] > 0).mean() >= 0.25]
        if not pair_firs:
            labels_final.loc[od_grp.index] = 0
            diagnostics.append({'ADEP': adep, 'ADES': ades,
                                 'n_flights': len(od_grp), 'n_clusters': 1})
            continue

        bin_X = (od_grp[pair_firs] > 0).astype(int).values
        k2    = _best_k(bin_X)
        l2    = (np.zeros(len(od_grp), dtype=int) if k2 == 1
                 else KMeans(n_clusters=k2, random_state=158, n_init=3).fit_predict(bin_X))

        cluster_counter = 0
        for l2_id in sorted(set(l2)):
            mask    = l2 == l2_id
            sub_idx = od_grp.index[mask]
            sub_X   = StandardScaler().fit_transform(
                          od_grp.loc[sub_idx, pair_firs].fillna(0).values)
            k3  = _best_k(sub_X)
            l3  = (np.zeros(mask.sum(), dtype=int) if k3 == 1
                   else KMeans(n_clusters=k3, random_state=158, n_init=3).fit_predict(sub_X))
            for l3_id in sorted(set(l3)):
                labels_final.loc[sub_idx[l3 == l3_id]] = cluster_counter
                cluster_counter += 1

        diagnostics.append({'ADEP': adep, 'ADES': ades,
                             'n_flights': len(od_grp), 'n_clusters': cluster_counter})

    return labels_final, pd.DataFrame(diagnostics)


if os.path.exists(FULL_LABELS_CACHE) and os.path.exists(FULL_DIAG_CACHE):
    _labels          = pd.read_csv(FULL_LABELS_CACHE)
    df_full          = df_full.merge(_labels, on='ECTRL ID', how='left')
    clust_diagnostic = pd.read_csv(FULL_DIAG_CACHE)
    print(f'Loaded from cache: {len(_labels):,} labels, {len(clust_diagnostic):,} pairs')
else:
    print(f'Clustering {n_pairs_full:,} pairs - expect 15-30 min...')
    full_labels, clust_diagnostic = cluster_full_dataset(df_full, _fir_all_cols)
    df_full['cluster_kmeans']     = full_labels
    df_full[['ECTRL ID', 'cluster_kmeans']].to_csv(FULL_LABELS_CACHE, index=False)
    clust_diagnostic.to_csv(FULL_DIAG_CACHE, index=False)
    print(f'\nDone. Labels saved.')

print('\nCluster count distribution across all pairs:')
print(clust_diagnostic['n_clusters'].value_counts().sort_index().rename('n_pairs').to_string())


In [ ]:
# Cell 31 - diagnostic: which pairs produce meaningful clustering?

import plotly.express as px

fig = px.scatter(
    clust_diagnostic.sample(min(5000, len(clust_diagnostic)), random_state=158),
    x='n_flights', y='n_clusters',
    opacity=0.35, log_x=True,
    color='n_clusters', color_continuous_scale='Viridis',
    title='Final cluster count vs flights per O-D pair',
    labels={'n_flights': 'flights per pair', 'n_clusters': 'final clusters (k)'},
)
fig.update_traces(marker_size=4)
fig.update_layout(height=450, coloraxis_showscale=False)
fig.show()

bands = [(30, 50), (50, 75), (75, 100), (100, 150), (150, 200), (200, 300), (300, 500), (500, 9999)]
print(f'\n{"Band":>12}  {"Pairs":>7}  {"k=1":>7}  {"k≥2":>7}  {"% k≥2":>7}')
print('-' * 48)
for lo, hi in bands:
    sub = clust_diagnostic[
        (clust_diagnostic['n_flights'] >= lo) & (clust_diagnostic['n_flights'] < hi)
    ]
    if sub.empty:
        continue
    k2p = (sub['n_clusters'] >= 2).sum()
    lbl = f'{lo}–{hi}' if hi < 9999 else f'{lo}+'
    print(f'{lbl:>12}  {len(sub):>7,}  {(sub["n_clusters"]==1).sum():>7,}  {k2p:>7,}  {k2p/len(sub)*100:>6.1f}%')

In [ ]:
# Cell 32 - full_summary: vectorised cost computation + cluster summary across all pairs
# cached: skips the 639k-row aggregation on rerun once full_summary/full_summary_ac exist

FULL_SUMMARY_CACHE    = '/drive/MyDrive/flight-project/full_summary.csv'
FULL_SUMMARY_AC_CACHE = '/drive/MyDrive/flight-project/full_summary_ac.csv'
MIN_N_AC = 10

if os.path.exists(FULL_SUMMARY_CACHE) and os.path.exists(FULL_SUMMARY_AC_CACHE):
    full_summary    = pd.read_csv(FULL_SUMMARY_CACHE)
    full_summary_ac = pd.read_csv(FULL_SUMMARY_AC_CACHE)
    print(f'Loaded from cache: full_summary ({len(full_summary):,} rows), '
          f'full_summary_ac ({len(full_summary_ac):,} rows)')
else:
    _ac_col_full = next(c for c in df_full.columns if 'AC Type' in c)
    ac_col       = _ac_col_full  # flight_fuel_eur reads this from outer scope

    RATE_FIRS = [f for f in list(EUROCONTROL_RATES.keys()) + ['CZQMFIR', 'CZULFIR', 'CZQXFIR']
                 if f in df_full.columns]

    df_full['mtow_t']     = df_full[_ac_col_full].map(MTOW_TONNES)
    df_full['duration_h'] = df_full['Duration_Hours'].apply(_parse_duration)

    def _atc_vec(df):
        mtow  = df['mtow_t']
        total = pd.Series(0.0, index=df.index)
        # Sum FIR + UIR per ANSP base code before charging
        base_totals = {}
        base_rate_map = {}
        for fir, rate in EUROCONTROL_RATES.items():
            if fir not in df.columns:
                continue
            base = fir[:-3]
            base_totals[base] = base_totals.get(base, pd.Series(0.0, index=df.index)) + df[fir].fillna(0)
            if base not in base_rate_map:
                base_rate_map[base] = rate
        for base, dist_nm_col in base_totals.items():
            dist_km = dist_nm_col * 1.852
            total  += (dist_km / 100) * np.sqrt(mtow / 50) * base_rate_map[base]
        for fir in ('CZQMFIR', 'CZULFIR'):
            if fir not in df.columns:
                continue
            dist_km = df[fir].fillna(0) * 1.852
            total  += NAV_CANADA_R * np.sqrt(mtow) * dist_km * CAD_EUR
        if 'CZQXFIR' in df.columns:
            total += (df['CZQXFIR'].fillna(0) > 0).astype(float) * NAV_CANADA_OCEANIC * CAD_EUR
        return total.where(mtow.notna()).round(2)

    def _fuel_vec(df, ac_col_name):
        fuel_kgh = df[ac_col_name].map(FUEL_KGH)
        return (fuel_kgh * df['duration_h'] * JET_A_EUR_PER_KG).where(
            fuel_kgh.notna() & df['mtow_t'].notna()
        ).round(2)

    df_full['atc_eur']  = _atc_vec(df_full)
    df_full['fuel_eur'] = _fuel_vec(df_full, _ac_col_full)
    df_full['cost_eur'] = (df_full['atc_eur'] + df_full['fuel_eur']).round(2)
    _dist_firs = [c for c in RATE_FIRS if c.endswith('FIR') and c != 'CZQXFIR']
    df_full['total_dist_nm'] = df_full[_dist_firs].fillna(0).sum(axis=1)

    print(f'Missing MTOW: {df_full["mtow_t"].isna().sum():,} | Missing cost: {df_full["cost_eur"].isna().sum():,}')

    agg_dict = {
        'n_flights':       ('ECTRL ID', 'count'),
        'mean_duration_h': ('duration_h', 'mean'),
        'most_common_ac':      (_ac_col_full, lambda x: x.mode().iloc[0]),
        'mean_total_dist_nm':  ('total_dist_nm', 'mean'),
    }
    agg_dict.update({f'mean_{f}': (f, 'mean') for f in RATE_FIRS})

    full_summary = df_full.groupby(['ADEP', 'ADES', 'cluster_kmeans']).agg(**agg_dict).reset_index()

    cost_agg_full = (
        df_full.dropna(subset=['cost_eur'])
        .groupby(['ADEP', 'ADES', 'cluster_kmeans'])['cost_eur']
        .agg(mean_cost_eur='mean', std_cost_eur='std').round(2).reset_index()
    )
    full_summary = full_summary.merge(cost_agg_full, on=['ADEP', 'ADES', 'cluster_kmeans'], how='left')
    full_summary['n_clusters_od'] = (
        full_summary.groupby(['ADEP', 'ADES'])['cluster_kmeans'].transform('nunique')
    )

    ac_agg_dict = {
        'n_flights_ac':          ('ECTRL ID', 'count'),
        'mean_duration_h_ac':    ('duration_h', 'mean'),
        'mean_total_dist_nm_ac': ('total_dist_nm', 'mean'),
    }
    ac_agg_dict.update({f'mean_{f}_ac': (f, 'mean') for f in RATE_FIRS})

    full_summary_ac = (
        df_full.groupby(['ADEP', 'ADES', 'cluster_kmeans', _ac_col_full])
        .agg(**ac_agg_dict)
        .reset_index()
        .rename(columns={_ac_col_full: 'ac_type'})
    )

    full_summary.to_csv(FULL_SUMMARY_CACHE, index=False)
    full_summary_ac.to_csv(FULL_SUMMARY_AC_CACHE, index=False)
    print('Computed and cached full_summary + full_summary_ac')

n_od  = full_summary.groupby(['ADEP', 'ADES']).ngroups
n_alt = (full_summary.groupby(['ADEP', 'ADES'])['cluster_kmeans'].nunique() >= 2).sum()
n_ac_specific = (full_summary_ac['n_flights_ac'] >= MIN_N_AC).sum()
print()
print(f'full_summary: {len(full_summary):,} cluster rows, {n_od:,} O-D pairs')
print(f'Pairs with alternatives (k>=2): {n_alt:,} ({n_alt / n_od * 100:.1f}%)')
print(f'full_summary_ac: {len(full_summary_ac):,} (cluster, ac_type) rows, '
      f'{n_ac_specific:,} meet MIN_N_AC={MIN_N_AC} for aircraft-specific lookup')


In [ ]:
# Cell 33 - updated predict_route_options; uses full_summary by default, with an
# aircraft-specific L3 lookup (full_summary_ac) that falls back to the pooled
# cluster mean when the queried ac_type has fewer than min_n_ac historical flights

def predict_route_options(ac_type, adep, ades, sort_by='cost_eur',
                           lookup=None, lookup_ac=None, min_n_ac=MIN_N_AC):
    src    = lookup if lookup is not None else full_summary
    src_ac = lookup_ac if lookup_ac is not None else full_summary_ac
    od_rows = src[(src['ADEP'] == adep) & (src['ADES'] == ades)]
    if od_rows.empty:
        raise ValueError(f'No clusters found for {adep}-{ades}')

    mtow = MTOW_TONNES.get(ac_type)
    if mtow is None:
        raise ValueError(f"'{ac_type}' not in MTOW_TONNES")
    fuel_kgh = FUEL_KGH.get(ac_type)
    if fuel_kgh is None:
        raise ValueError(f"'{ac_type}' not in FUEL_KGH")

    ac_rows = src_ac[
        (src_ac['ADEP'] == adep) & (src_ac['ADES'] == ades) & (src_ac['ac_type'] == ac_type)
    ].set_index('cluster_kmeans')

    results = []
    for _, row in od_rows.iterrows():
        cluster = row['cluster_kmeans']
        ac_row  = ac_rows.loc[cluster] if cluster in ac_rows.index else None
        ac_specific = ac_row is not None and ac_row['n_flights_ac'] >= min_n_ac

        if ac_specific:
            duration = ac_row['mean_duration_h_ac']
            rep_row  = {fir: ac_row.get(f'mean_{fir}_ac', 0)
                        for fir in list(EUROCONTROL_RATES) + ['CZQMFIR', 'CZULFIR', 'CZQXFIR']}
            dist_nm  = ac_row.get('mean_total_dist_nm_ac')
            n_hist   = int(ac_row['n_flights_ac'])
        else:
            duration = row['mean_duration_h']
            rep_row  = {fir: row.get(f'mean_{fir}', 0)
                        for fir in list(EUROCONTROL_RATES) + ['CZQMFIR', 'CZULFIR', 'CZQXFIR']}
            dist_nm  = row.get('mean_total_dist_nm')
            n_hist   = int(row['n_flights'])

        rep_row['mtow_t'] = mtow
        pred_atc  = round(flight_atc_eur(rep_row), 2)
        pred_fuel = round(fuel_kgh * duration * JET_A_EUR_PER_KG, 2)

        results.append({
            'cluster':              cluster,
            'ac_specific':          ac_specific,
            'n_historical_flights': n_hist,
            'mean_dist_nm':         round(float(dist_nm), 1) if pd.notna(dist_nm) else None,
            'mean_duration_h':      round(duration, 3),
            'predicted_atc_eur':    pred_atc,
            'predicted_fuel_eur':   pred_fuel,
            'predicted_cost_eur':   round(pred_atc + pred_fuel, 2),
        })

    sort_col = 'predicted_cost_eur' if sort_by == 'cost_eur' else 'mean_duration_h'
    df_out = pd.DataFrame(results).sort_values(sort_col).reset_index(drop=True)
    df_out.index += 1
    df_out.index.name = 'rank'
    return df_out


print('A35K | EGLL-KJFK')
display(predict_route_options('A35K', 'EGLL', 'KJFK'))
print('\nA320 | LEBL-LEPA')
display(predict_route_options('A320', 'LEBL', 'LEPA'))
print('\nA320 | LPPT-EDDB')
display(predict_route_options('A320', 'LPPT', 'EDDB'))

In [ ]:
# Cell 34 - load actual + filed trajectory data, filter to training sample
# Upload to /drive/MyDrive/flight-project/ before running:
#   Flight_FIRs_Actual_20230901_20230930.csv
#   Flight_Points_Actual_20230901_20230930.csv
#   Flight_Points_Filed_20230901_20230930.csv

ACTUAL_FIRS_PATH    = '/drive/MyDrive/flight-project/Flight_FIRs_Actual_20230901_20230930.csv'
ACTUAL_POINTS_PATH  = '/drive/MyDrive/flight-project/Flight_Points_Actual_20230901_20230930.csv'
FILED_POINTS_PATH   = '/drive/MyDrive/flight-project/Flight_Points_Filed_20230901_20230930.csv'
ACTUAL_FIRS_CACHE   = '/drive/MyDrive/flight-project/actual_firs_sample.csv'
ACTUAL_POINTS_CACHE = '/drive/MyDrive/flight-project/actual_points_sample.csv'
FILED_POINTS_CACHE  = '/drive/MyDrive/flight-project/filed_points_sample.csv'

sample_ids = set(df_sample['ECTRL ID'].astype(str))

if os.path.exists(ACTUAL_FIRS_CACHE) and os.path.exists(ACTUAL_POINTS_CACHE):
    actual_firs = pd.read_csv(ACTUAL_FIRS_CACHE, dtype={'ECTRL ID': str})
    actual_pts  = pd.read_csv(ACTUAL_POINTS_CACHE, dtype={'ECTRL ID': str})
    print('Actual data loaded from cache')
else:
    _raw = pd.read_csv(ACTUAL_FIRS_PATH, dtype={'ECTRL ID': str})
    actual_firs = _raw[_raw['ECTRL ID'].isin(sample_ids)].copy().reset_index(drop=True)
    del _raw
    actual_firs.to_csv(ACTUAL_FIRS_CACHE, index=False)
    _chunks = []
    for _chunk in pd.read_csv(ACTUAL_POINTS_PATH, dtype={'ECTRL ID': str}, chunksize=500_000):
        _chunks.append(_chunk[_chunk['ECTRL ID'].isin(sample_ids)])
    actual_pts = pd.concat(_chunks, ignore_index=True)
    del _chunks
    actual_pts.to_csv(ACTUAL_POINTS_CACHE, index=False)
    print('Actual data filtered and cached')

if os.path.exists(FILED_POINTS_CACHE):
    filed_pts = pd.read_csv(FILED_POINTS_CACHE, dtype={'ECTRL ID': str})
    print('Filed points loaded from cache')
else:
    _chunks = []
    for _chunk in pd.read_csv(FILED_POINTS_PATH, dtype={'ECTRL ID': str}, chunksize=500_000):
        _chunks.append(_chunk[_chunk['ECTRL ID'].isin(sample_ids)])
    filed_pts = pd.concat(_chunks, ignore_index=True)
    del _chunks
    filed_pts.to_csv(FILED_POINTS_CACHE, index=False)
    print('Filed points filtered and cached')

print(f'Actual FIR rows : {len(actual_firs):,}  |  flights: {actual_firs["ECTRL ID"].nunique():,}')
print(f'Actual point rows: {len(actual_pts):,}')
print(f'Filed point rows : {len(filed_pts):,}')
print(f'Sample flights missing from actual: {len(sample_ids - set(actual_firs["ECTRL ID"])):,}')


In [ ]:
# Cell 35 - actual metrics + planned_dist_nm (haversine from filed trajectory) per flight
# cached: skips the per-flight Python loop below (unvectorised, the slowest part of this
# notebook) once df_actual_sample.csv exists

from math import radians, sin, cos, sqrt, atan2

DF_ACTUAL_CACHE = '/drive/MyDrive/flight-project/df_actual_sample.csv'

def haversine_nm(lat1, lon1, lat2, lon2):
    R = 3440.065
    lat1, lon1, lat2, lon2 = map(radians, [float(lat1), float(lon1), float(lat2), float(lon2)])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


def _flight_actual_metrics(firs_grp, pts_grp):
    """Returns (fir_distances dict nm, actual_duration_h, actual_total_dist_nm)."""
    f = firs_grp.copy()
    f['entry_dt'] = pd.to_datetime(f['Entry Time'], dayfirst=True)
    f['exit_dt']  = pd.to_datetime(f['Exit Time'],  dayfirst=True)
    airborne = (f[~f['FIR ID'].isin(['TAXI_OUT', 'TAXI_IN'])]
                .sort_values('entry_dt').reset_index(drop=True))
    if airborne.empty:
        return {}, np.nan, 0.0

    dur_h = (airborne['exit_dt'].iloc[-1] - airborne['entry_dt'].iloc[0]).total_seconds() / 3600

    p = pts_grp.copy()
    p['time_dt'] = pd.to_datetime(p['Time Over'], dayfirst=True)
    p = (p[(p['time_dt'] >= airborne['entry_dt'].iloc[0]) &
           (p['time_dt'] <= airborne['exit_dt'].iloc[-1])]
         .sort_values('time_dt').reset_index(drop=True))
    if len(p) < 2:
        return {}, dur_h, 0.0

    entries = airborne['entry_dt'].values.astype('datetime64[ns]')
    exits   = airborne['exit_dt'].values.astype('datetime64[ns]')
    fids    = airborne['FIR ID'].values
    lats    = p['Latitude'].values.astype(float)
    lons    = p['Longitude'].values.astype(float)
    times   = p['time_dt'].values.astype('datetime64[ns]')

    def _idx(t):
        for i in range(len(entries)):
            if entries[i] <= t <= exits[i]:
                return i
        return -1

    fir_dists, total_dist = {}, 0.0
    for i in range(len(p) - 1):
        d = haversine_nm(lats[i], lons[i], lats[i + 1], lons[i + 1])
        total_dist += d
        i1, i2 = _idx(times[i]), _idx(times[i + 1])
        if i1 == i2:
            if i1 >= 0:
                fir_dists[fids[i1]] = fir_dists.get(fids[i1], 0) + d
        elif i1 >= 0:
            dt = float((times[i + 1] - times[i]) / np.timedelta64(1, 's'))
            if dt > 0:
                frac = max(0.0, min(1.0,
                    float((exits[i1] - times[i]) / np.timedelta64(1, 's')) / dt))
                fir_dists[fids[i1]] = fir_dists.get(fids[i1], 0) + d * frac
                if i2 >= 0:
                    fir_dists[fids[i2]] = fir_dists.get(fids[i2], 0) + d * (1 - frac)
    # Collapse UIR into FIR: FIR and UIR of the same ANSP are one billing zone
    for fid in list(fir_dists.keys()):
        if fid.endswith('UIR'):
            base_fir = fid[:-3] + 'FIR'
            if base_fir in EUROCONTROL_RATES:
                fir_dists[base_fir] = fir_dists.get(base_fir, 0) + fir_dists.pop(fid)
    return fir_dists, dur_h, total_dist


def _total_dist_from_pts(pts_grp):
    """Haversine sum over consecutive filed trajectory points."""
    p    = pts_grp.sort_values('Sequence Number').reset_index(drop=True)
    lats = p['Latitude'].values.astype(float)
    lons = p['Longitude'].values.astype(float)
    total = 0.0
    for i in range(len(p) - 1):
        total += haversine_nm(lats[i], lons[i], lats[i + 1], lons[i + 1])
    return round(total, 1)


if os.path.exists(DF_ACTUAL_CACHE):
    df_actual = pd.read_csv(DF_ACTUAL_CACHE)
    print(f'Loaded from cache: {len(df_actual):,} flights')
else:
    _firs_by_id  = dict(list(actual_firs.groupby('ECTRL ID')))
    _pts_by_id   = dict(list(actual_pts.groupby('ECTRL ID')))
    _filed_by_id = dict(list(filed_pts.groupby('ECTRL ID')))
    _ac_col      = next(c for c in df_sample.columns if 'AC Type' in c)
    _meta        = df_sample[['ECTRL ID', _ac_col, 'mtow_t']].copy()
    _meta['ECTRL ID'] = _meta['ECTRL ID'].astype(str)

    records, skipped = [], []
    for _, mrow in _meta.iterrows():
        eid = str(mrow['ECTRL ID'])
        if eid not in _firs_by_id or eid not in _pts_by_id:
            skipped.append(eid)
            continue
        fir_dists, dur_h, total_dist = _flight_actual_metrics(_firs_by_id[eid], _pts_by_id[eid])
        if not fir_dists or pd.isna(dur_h):
            skipped.append(eid)
            continue
        ac, mtow = mrow[_ac_col], mrow['mtow_t']
        fkgh     = FUEL_KGH.get(ac)
        rep      = dict(fir_dists)
        rep['mtow_t'] = mtow
        atc_eur  = flight_atc_eur(rep) if not pd.isna(mtow) else np.nan
        fuel_eur = round(fkgh * dur_h * JET_A_EUR_PER_KG, 2) if fkgh else np.nan
        fuel_kg  = round(fkgh * dur_h, 1)                    if fkgh else np.nan
        cost_eur = round(atc_eur + fuel_eur, 2) if not (pd.isna(atc_eur) or pd.isna(fuel_eur)) else np.nan
        records.append({
            'ECTRL ID':             int(eid),
            'actual_duration_h':    round(dur_h, 4),
            'actual_total_dist_nm': round(total_dist, 1),
            'actual_atc_eur':       atc_eur,
            'actual_fuel_eur':      fuel_eur,
            'actual_fuel_kg':       fuel_kg,
            'actual_cost_eur':      cost_eur,
            'planned_dist_nm':      _total_dist_from_pts(_filed_by_id[eid]) if eid in _filed_by_id else np.nan,
        })

    df_actual = pd.DataFrame(records)
    df_actual.to_csv(DF_ACTUAL_CACHE, index=False)
    print(f'Computed: {len(df_actual):,}  |  skipped: {len(skipped):,}')

print(df_actual[['actual_duration_h', 'actual_total_dist_nm', 'planned_dist_nm',
                 'actual_atc_eur', 'actual_fuel_eur', 'actual_cost_eur']].describe().round(2))


In [ ]:
# Cell 36 - L3 centroids (planned) and per-flight deltas (actual - centroid)
# delta_cost_eur excluded: planned FIR double-counting inflates centroid_cost_eur vs actual
# cached: also caches df_compared itself (not just the summary tables), since downstream
# cells need it; this closes a gap where cell 43's loader expected error_summary /
# error_summary_pooled cache files that nothing previously wrote

L3_CACHE          = '/drive/MyDrive/flight-project/l3_centroids.csv'
L3_CACHE_POOLED   = '/drive/MyDrive/flight-project/l3_centroids_pooled.csv'
ERR_CACHE         = '/drive/MyDrive/flight-project/error_summary.csv'
ERR_CACHE_POOLED  = '/drive/MyDrive/flight-project/error_summary_pooled.csv'
DF_COMPARED_CACHE = '/drive/MyDrive/flight-project/df_compared.csv'

DELTA_COLS = ['delta_dist_nm', 'delta_duration_h', 'delta_fuel_kg']
RATE_FIRS_SAMPLE = [f for f in list(EUROCONTROL_RATES.keys()) + ['CZQMFIR', 'CZULFIR', 'CZQXFIR']
                     if f in df_sample.columns]  # needed by query_route_profile either way

_cache_paths = [L3_CACHE, L3_CACHE_POOLED, ERR_CACHE, ERR_CACHE_POOLED, DF_COMPARED_CACHE]

if all(os.path.exists(p) for p in _cache_paths):
    l3_centroids         = pd.read_csv(L3_CACHE)
    l3_centroids_pooled  = pd.read_csv(L3_CACHE_POOLED)
    error_summary        = pd.read_csv(ERR_CACHE)
    error_summary_pooled = pd.read_csv(ERR_CACHE_POOLED)
    df_compared          = pd.read_csv(DF_COMPARED_CACHE)
    _ac_col = next(c for c in l3_centroids.columns if 'AC Type' in c)
    print(f'Loaded from cache: l3_centroids ({len(l3_centroids)}), '
          f'l3_centroids_pooled ({len(l3_centroids_pooled)}), '
          f'error_summary ({len(error_summary)}), '
          f'error_summary_pooled ({len(error_summary_pooled)}), '
          f'df_compared ({len(df_compared)})')
else:
    _ac_col = next(c for c in df_sample.columns if 'AC Type' in c)

    df_sample['ECTRL ID']           = df_sample['ECTRL ID'].astype(int)
    df_sample['planned_duration_h'] = df_sample['Duration_Hours'].apply(_parse_duration)
    df_sample['planned_fuel_kg']    = (df_sample['fuel_eur'] / JET_A_EUR_PER_KG).round(1)

    df_compared = df_sample.merge(df_actual, on='ECTRL ID', how='left')

    CENTROID_FEATS = ['planned_dist_nm', 'planned_duration_h', 'planned_fuel_kg', 'cost_eur']

    l3_centroids = (
        df_compared.dropna(subset=CENTROID_FEATS)
        .groupby(['ADEP', 'ADES', 'cluster_kmeans', _ac_col])
        [CENTROID_FEATS]
        .agg(
            centroid_dist_nm   =('planned_dist_nm',    'mean'),
            centroid_duration_h=('planned_duration_h', 'mean'),
            centroid_fuel_kg   =('planned_fuel_kg',    'mean'),
            centroid_cost_eur  =('cost_eur',           'mean'),
            n_l3               =('planned_dist_nm',    'count'),
        )
        .round({'centroid_dist_nm': 1, 'centroid_duration_h': 3,
                'centroid_fuel_kg': 1, 'centroid_cost_eur': 2})
        .reset_index()
    )
    print('L3 centroids:')
    display(l3_centroids[['ADEP', 'ADES', 'cluster_kmeans', _ac_col, 'n_l3',
                           'centroid_dist_nm', 'centroid_duration_h',
                           'centroid_fuel_kg', 'centroid_cost_eur']])

    df_compared = df_compared.merge(
        l3_centroids[['ADEP', 'ADES', 'cluster_kmeans', _ac_col,
                      'centroid_dist_nm', 'centroid_duration_h',
                      'centroid_fuel_kg', 'centroid_cost_eur']],
        on=['ADEP', 'ADES', 'cluster_kmeans', _ac_col],
        how='left',
    )

    df_compared['delta_dist_nm']    = (df_compared['actual_total_dist_nm'] - df_compared['centroid_dist_nm']).round(1)
    df_compared['delta_duration_h'] = (df_compared['actual_duration_h']    - df_compared['centroid_duration_h']).round(4)
    df_compared['delta_fuel_kg']    = (df_compared['actual_fuel_kg']       - df_compared['centroid_fuel_kg']).round(1)

    error_rows = []
    for (adep, ades, clust, ac), grp in df_compared.dropna(subset=DELTA_COLS).groupby(
            ['ADEP', 'ADES', 'cluster_kmeans', _ac_col]):
        row = {'ADEP': adep, 'ADES': ades, 'cluster_kmeans': clust, _ac_col: ac, 'n': len(grp)}
        for col in DELTA_COLS:
            row[f'{col}_mean'] = round(grp[col].mean(), 2)
            row[f'{col}_std']  = round(grp[col].std(),  2)
            row[f'{col}_p5']   = round(grp[col].quantile(0.05), 2)
            row[f'{col}_p95']  = round(grp[col].quantile(0.95), 2)
        error_rows.append(row)

    error_summary = pd.DataFrame(error_rows)
    print('Error summary (actual vs centroid):')
    display(error_summary)

    _pooled_agg = {
        'centroid_dist_nm':    ('planned_dist_nm',    'mean'),
        'centroid_duration_h': ('planned_duration_h', 'mean'),
        'n_l3':                ('planned_dist_nm',    'count'),
    }
    _pooled_agg.update({f'mean_{f}': (f, 'mean') for f in RATE_FIRS_SAMPLE})

    l3_centroids_pooled = (
        df_compared.dropna(subset=CENTROID_FEATS)
        .groupby(['ADEP', 'ADES', 'cluster_kmeans'])
        .agg(**_pooled_agg)
        .round({'centroid_dist_nm': 1, 'centroid_duration_h': 3})
        .reset_index()
    )

    df_compared = df_compared.merge(
        l3_centroids_pooled[['ADEP', 'ADES', 'cluster_kmeans', 'centroid_dist_nm', 'centroid_duration_h']],
        on=['ADEP', 'ADES', 'cluster_kmeans'],
        how='left',
        suffixes=('', '_pooled'),
    )

    df_compared['delta_dist_nm_pooled']    = (df_compared['actual_total_dist_nm'] - df_compared['centroid_dist_nm_pooled']).round(1)
    df_compared['delta_duration_h_pooled'] = (df_compared['actual_duration_h']    - df_compared['centroid_duration_h_pooled']).round(4)

    DELTA_COLS_POOLED = ['delta_dist_nm_pooled', 'delta_duration_h_pooled']

    error_rows_pooled = []
    for (adep, ades, clust), grp in df_compared.dropna(subset=DELTA_COLS_POOLED).groupby(
            ['ADEP', 'ADES', 'cluster_kmeans']):
        row = {'ADEP': adep, 'ADES': ades, 'cluster_kmeans': clust, 'n': len(grp)}
        for col in DELTA_COLS_POOLED:
            base = col.replace('_pooled', '')
            row[f'{base}_mean'] = round(grp[col].mean(), 2)
            row[f'{base}_std']  = round(grp[col].std(),  2)
            row[f'{base}_p5']   = round(grp[col].quantile(0.05), 2)
            row[f'{base}_p95']  = round(grp[col].quantile(0.95), 2)
        error_rows_pooled.append(row)

    error_summary_pooled = pd.DataFrame(error_rows_pooled)
    print('Pooled error summary (actual vs pooled centroid; dist/duration only, fuel/cost recomputed per query):')
    display(error_summary_pooled)

    l3_centroids.to_csv(L3_CACHE, index=False)
    l3_centroids_pooled.to_csv(L3_CACHE_POOLED, index=False)
    error_summary.to_csv(ERR_CACHE, index=False)
    error_summary_pooled.to_csv(ERR_CACHE_POOLED, index=False)
    df_compared.to_csv(DF_COMPARED_CACHE, index=False)
    print(f'Cached: l3_centroids ({len(l3_centroids)}), l3_centroids_pooled ({len(l3_centroids_pooled)}), '
          f'error_summary ({len(error_summary)}), error_summary_pooled ({len(error_summary_pooled)}), '
          f'df_compared ({len(df_compared)})')


In [ ]:
# Cell 37 - delta distributions + representative actual trajectory per cluster

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

_ac_col = next(c for c in df_sample.columns if 'AC Type' in c)

# Delta distributions per O-D pair (distance, duration, fuel)
for (adep, ades), grp in df_compared.dropna(subset=DELTA_COLS).groupby(['ADEP', 'ADES']):
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=['\u0394 distance (nm)',
                                        '\u0394 duration (h)',
                                        '\u0394 fuel (kg)'])
    for col, c in [('delta_dist_nm', 1), ('delta_duration_h', 2), ('delta_fuel_kg', 3)]:
        fig.add_trace(go.Histogram(x=grp[col].dropna(), nbinsx=30,
                                    marker_color='#4C78A8', showlegend=False),
                      row=1, col=c)
        fig.add_vline(x=0, line_dash='dash', line_color='#E45756', row=1, col=c)
    fig.update_layout(title_text=f'{adep}-{ades}: actual \u2212 L3 centroid',
                      height=380, margin=dict(t=80))
    fig.show()


def _representative_id(grp_df, cen):
    feats = ['planned_dist_nm', 'planned_duration_h', 'planned_fuel_kg', 'cost_eur']
    cent  = np.array([cen['centroid_dist_nm'], cen['centroid_duration_h'],
                      cen['centroid_fuel_kg'],  cen['centroid_cost_eur']])
    sub = grp_df[feats].dropna()
    if sub.empty:
        return None
    diffs = sub.values - cent
    scale = np.abs(diffs).max(axis=0) + 1e-9
    return grp_df.loc[sub.iloc[np.linalg.norm(diffs / scale, axis=1).argmin()].name, 'ECTRL ID']


# Actual trajectory of representative flight per (OD pair, cluster)
for (adep, ades), od_grp in df_compared.groupby(['ADEP', 'ADES']):
    for clust in sorted(od_grp['cluster_kmeans'].unique()):
        c_grp  = od_grp[od_grp['cluster_kmeans'] == clust]
        top_ac = c_grp[_ac_col].value_counts().idxmax()
        cen    = l3_centroids[
            (l3_centroids['ADEP'] == adep) & (l3_centroids['ADES'] == ades) &
            (l3_centroids['cluster_kmeans'] == clust) & (l3_centroids[_ac_col] == top_ac)
        ]
        if cen.empty:
            continue
        rep_id = _representative_id(c_grp[c_grp[_ac_col] == top_ac], cen.iloc[0])
        if rep_id is None:
            continue

        pts = actual_pts[actual_pts['ECTRL ID'] == str(rep_id)].copy()
        pts['time_dt'] = pd.to_datetime(pts['Time Over'], dayfirst=True)
        pts = pts.sort_values('time_dt')
        if pts.empty:
            continue

        dr     = df_compared[df_compared['ECTRL ID'] == rep_id]
        d_dist = dr['delta_dist_nm'].values[0]    if not dr.empty else np.nan
        d_dur  = dr['delta_duration_h'].values[0] if not dr.empty else np.nan
        d_dist_str = f'{d_dist:+.0f} nm'   if not pd.isna(d_dist) else 'N/A'
        d_dur_str  = f'{d_dur:+.3f} h'     if not pd.isna(d_dur)  else 'N/A'

        fig = go.Figure(go.Scattergeo(
            lat=pts['Latitude'].astype(float),
            lon=pts['Longitude'].astype(float),
            mode='lines+markers',
            line=dict(width=2, color='#4C78A8'),
            marker=dict(size=3),
        ))
        fig.update_geos(
            projection_type='natural earth', fitbounds='locations',
            showland=True, landcolor='#E8E8E8',
            showocean=True, oceancolor='#C9DEF4',
            showcoastlines=True, coastlinecolor='#999999',
        )
        fig.update_layout(
            title=(f'{adep}-{ades} | Cluster {clust} representative actual trajectory<br>'
                   f'ECTRL {rep_id} ({top_ac}) \u2014 '
                   f'\u0394dist {d_dist_str} | \u0394dur {d_dur_str}'),
            height=450, margin=dict(t=80),
        )
        fig.show()


In [ ]:
# Cell 39b - noise distribution: actual flights vs their cluster+AC-type centroid
# One figure per O-D pair; rows = cluster, columns = metric (dist/duration/fuel);
# AC type shown as a colour split within each row since L3 already splits by AC type.

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

def plot_noise_distribution(df, ac_col=None, label_col='cluster_kmeans', delta_cols=DELTA_COLS):
    if ac_col is None:
        ac_col = next(c for c in df.columns if 'AC Type' in c)

    df = df.dropna(subset=delta_cols)

    for (adep, ades), grp in df.groupby(['ADEP', 'ADES']):
        clusters   = sorted(grp[label_col].unique())
        ac_types   = sorted(grp[ac_col].unique())
        ac_colours = {ac: px.colors.qualitative.Plotly[i % 10] for i, ac in enumerate(ac_types)}

        fig = make_subplots(
            rows=len(clusters), cols=len(delta_cols),
            subplot_titles=[f'C{c} | {col}' for c in clusters for col in delta_cols],
        )

        seen_ac = set()
        for r, cluster in enumerate(clusters, start=1):
            cluster_grp = grp[grp[label_col] == cluster]

            for c_idx, col in enumerate(delta_cols, start=1):
                for ac in sorted(cluster_grp[ac_col].unique()):
                    ac_vals = cluster_grp.loc[cluster_grp[ac_col] == ac, col].dropna()
                    if ac_vals.empty:
                        continue
                    fig.add_trace(
                        go.Histogram(
                            x=ac_vals, name=ac, nbinsx=20,
                            marker_color=ac_colours[ac],
                            legendgroup=ac, showlegend=(ac not in seen_ac),
                        ),
                        row=r, col=c_idx,
                    )
                    seen_ac.add(ac)
                fig.add_vline(x=0, line_dash='dash', line_color='#E45756', row=r, col=c_idx)

        fig.update_layout(
            title_text=f'{adep}-{ades}: noise (actual - centroid) by cluster and AC type',
            height=max(300, len(clusters) * 220),
            barmode='overlay',
            margin=dict(t=100),
        )
        fig.update_traces(opacity=0.65)
        fig.show()


plot_noise_distribution(df_compared)


In [ ]:
# Cell 39c - self-comparison: each flight's own actual vs its own planned (not vs cluster centroid)
# Different question from cells 36-39b: how well the filed plan predicts what that same
# flight actually flew, regardless of which cluster it landed in. High values here for a
# cluster/pair mean route planning is less reliable there, independent of the simulator's
# cluster-representative noise model (cells 36-39b).
# delta_cost_eur_self excluded: same planned-FIR-double-counting caveat as cell 36's delta_cost_eur.

SELF_ERR_CACHE      = '/drive/MyDrive/flight-project/error_summary_self.csv'
SELF_ERR_PAIR_CACHE = '/drive/MyDrive/flight-project/error_summary_self_by_pair.csv'

DELTA_COLS_SELF = ['delta_dist_nm_self', 'delta_duration_h_self', 'delta_fuel_kg_self']

if os.path.exists(SELF_ERR_CACHE) and os.path.exists(SELF_ERR_PAIR_CACHE):
    error_summary_self      = pd.read_csv(SELF_ERR_CACHE)
    error_summary_self_pair = pd.read_csv(SELF_ERR_PAIR_CACHE)
    print(f'Loaded from cache: error_summary_self ({len(error_summary_self)}), '
          f'error_summary_self_by_pair ({len(error_summary_self_pair)})')
else:
    df_compared['delta_dist_nm_self']    = (df_compared['actual_total_dist_nm'] - df_compared['planned_dist_nm']).round(1)
    df_compared['delta_duration_h_self'] = (df_compared['actual_duration_h']    - df_compared['planned_duration_h']).round(4)
    df_compared['delta_fuel_kg_self']    = (df_compared['actual_fuel_kg']       - df_compared['planned_fuel_kg']).round(1)

    def _summarise_self(df, group_cols):
        rows = []
        for key, grp in df.dropna(subset=DELTA_COLS_SELF).groupby(group_cols):
            key_tuple = key if isinstance(key, tuple) else (key,)
            row = dict(zip(group_cols, key_tuple))
            row['n'] = len(grp)
            for col in DELTA_COLS_SELF:
                row[f'{col}_mean'] = round(grp[col].mean(), 2)
                row[f'{col}_std']  = round(grp[col].std(),  2)
                row[f'{col}_p5']   = round(grp[col].quantile(0.05), 2)
                row[f'{col}_p95']  = round(grp[col].quantile(0.95), 2)
            rows.append(row)
        return pd.DataFrame(rows)

    error_summary_self      = _summarise_self(df_compared, ['ADEP', 'ADES', 'cluster_kmeans'])
    error_summary_self_pair = _summarise_self(df_compared, ['ADEP', 'ADES'])

    error_summary_self.to_csv(SELF_ERR_CACHE, index=False)
    error_summary_self_pair.to_csv(SELF_ERR_PAIR_CACHE, index=False)
    print(f'Cached: error_summary_self ({len(error_summary_self)}), '
          f'error_summary_self_by_pair ({len(error_summary_self_pair)})')

print('Self-comparison (actual - own planned), per cluster:')
display(error_summary_self)
print('Self-comparison (actual - own planned), per O-D pair (pooled across clusters):')
display(error_summary_self_pair)

plot_noise_distribution(df_compared, delta_cols=DELTA_COLS_SELF)


In [ ]:
# Cell 38 -- query function: L3 centroid + error envelope per (OD pair, ac type)
# Depends on: l3_centroids, error_summary, l3_centroids_pooled, error_summary_pooled, _ac_col
# (cells 36-37 + cell 40 outlier audit). Falls back to the pooled cluster when the requested
# ac_type has fewer than min_n_ac flights there. Pooled distance/duration are empirical;
# pooled fuel/cost are recomputed via the cost formula for the queried aircraft's own
# MTOW/fuel burn (blended historical fuel/cost across aircraft types is not meaningful --
# see cell 36) -- so delta_fuel_kg_* is not reported for pooled rows, only dist/duration.
# Clusters with n_l3 < 10 are flagged unreliable.

def query_route_profile(ac_type, adep, ades, sort_by='cost_eur', min_n_ac=MIN_N_AC):
    pooled = l3_centroids_pooled[
        (l3_centroids_pooled['ADEP'] == adep) & (l3_centroids_pooled['ADES'] == ades)
    ]
    if pooled.empty:
        raise ValueError(
            f'No L3 data for {adep}-{ades}. '
            f'Run the clustering pipeline for this OD pair first, '
            f'or use predict_route_options for the full dataset.'
        )

    mtow = MTOW_TONNES.get(ac_type)
    if mtow is None:
        raise ValueError(f"'{ac_type}' not in MTOW_TONNES")
    fuel_kgh = FUEL_KGH.get(ac_type)
    if fuel_kgh is None:
        raise ValueError(f"'{ac_type}' not in FUEL_KGH")

    ac_cen = l3_centroids[
        (l3_centroids['ADEP'] == adep) &
        (l3_centroids['ADES'] == ades) &
        (l3_centroids[_ac_col] == ac_type)
    ].set_index('cluster_kmeans')

    delta_cols = [c for c in error_summary.columns if c.startswith('delta_')]

    ac_err = error_summary[
        (error_summary['ADEP'] == adep) &
        (error_summary['ADES'] == ades) &
        (error_summary[_ac_col] == ac_type)
    ].set_index('cluster_kmeans')

    pooled_err = error_summary_pooled[
        (error_summary_pooled['ADEP'] == adep) & (error_summary_pooled['ADES'] == ades)
    ].set_index('cluster_kmeans')

    rows = []
    for _, prow in pooled.iterrows():
        cluster = prow['cluster_kmeans']
        ac_row  = ac_cen.loc[cluster] if cluster in ac_cen.index else None
        ac_specific = ac_row is not None and ac_row['n_l3'] >= min_n_ac

        if ac_specific:
            n_l3     = int(ac_row['n_l3'])
            dist_nm  = ac_row['centroid_dist_nm']
            duration = ac_row['centroid_duration_h']
            fuel_kg  = ac_row['centroid_fuel_kg']
            cost_eur = ac_row['centroid_cost_eur']
            err_row  = ac_err.loc[cluster] if cluster in ac_err.index else None
        else:
            n_l3     = int(prow['n_l3'])
            dist_nm  = prow['centroid_dist_nm']
            duration = prow['centroid_duration_h']
            rep_row  = {fir: prow.get(f'mean_{fir}', 0) for fir in RATE_FIRS_SAMPLE}
            rep_row['mtow_t'] = mtow
            fuel_kg  = round(fuel_kgh * duration, 1)
            cost_eur = round(flight_atc_eur(rep_row) + fuel_kg * JET_A_EUR_PER_KG, 2)
            err_row  = pooled_err.loc[cluster] if cluster in pooled_err.index else None

        row = {
            'cluster_kmeans':      cluster,
            'ac_specific':         ac_specific,
            'n_l3':                n_l3,
            'centroid_dist_nm':    dist_nm,
            'centroid_duration_h': duration,
            'centroid_fuel_kg':    fuel_kg,
            'centroid_cost_eur':   cost_eur,
        }
        for c in delta_cols:
            if not ac_specific and c.startswith('delta_fuel_kg'):
                row[c] = np.nan
            elif err_row is not None and c in err_row.index:
                row[c] = err_row[c]
            else:
                row[c] = np.nan
        rows.append(row)

    merged = pd.DataFrame(rows)
    merged['reliable'] = merged['n_l3'] >= 10

    _SORT = {
        'cost_eur':   'centroid_cost_eur',
        'duration_h': 'centroid_duration_h',
        'dist_nm':    'centroid_dist_nm',
    }
    sort_col = _SORT.get(sort_by, sort_by)

    out_cols = [
        'cluster_kmeans', 'ac_specific', 'n_l3', 'reliable',
        'centroid_dist_nm',    'delta_dist_nm_mean',    'delta_dist_nm_std',    'delta_dist_nm_p5',    'delta_dist_nm_p95',
        'centroid_duration_h', 'delta_duration_h_mean', 'delta_duration_h_std', 'delta_duration_h_p5', 'delta_duration_h_p95',
        'centroid_fuel_kg',    'delta_fuel_kg_mean',    'delta_fuel_kg_std',    'delta_fuel_kg_p5',    'delta_fuel_kg_p95',
        'centroid_cost_eur',
    ]
    df_out = (
        merged[[c for c in out_cols if c in merged.columns]]
        .sort_values(sort_col)
        .reset_index(drop=True)
    )
    df_out.index += 1
    df_out.index.name = 'rank'

    n_unreliable = (~df_out['reliable']).sum()
    if n_unreliable:
        print(f'Warning: {n_unreliable} cluster(s) with n_l3 < 10 -- error stats unreliable (see reliable column)')

    n_pooled = (~df_out['ac_specific']).sum()
    if n_pooled:
        print(f"Note: {n_pooled} cluster(s) used the pooled route with '{ac_type}'-specific fuel/cost "
              f"recomputed via formula -- '{ac_type}' had fewer than {min_n_ac} flights there")

    delta_present = [c for c in df_out.columns if c.startswith('delta_')]
    if not delta_present or df_out[delta_present].isna().all().all():
        print(f'Note: error stats not computed for {adep}-{ades}. '
              f'Centroid returned only. Run cells 34-40 for this OD pair to add error bounds.')

    return df_out

In [ ]:
# Cell 39 -- example queries for query_route_profile

print('A35K | EGLL-KJFK | ranked by cost')
display(query_route_profile('A35K', 'EGLL', 'KJFK', sort_by='cost_eur'))

print('\nB738 | LEBL-LEPA')
display(query_route_profile('B738', 'LEBL', 'LEPA'))

print('\nA320 | LPPT-EDDB')
display(query_route_profile('A320', 'LPPT', 'EDDB'))


In [ ]:
# Cell 40 -- outlier audit: rebuild error_summary using MAD-based outlier removal
# Modified z-score (Iglewicz & Hoaglin) with threshold 3.5 -- robust at small n

def _mad_filter(vals):
    arr = np.asarray(vals, dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return arr, 0
    med = np.median(arr)
    mad = np.median(np.abs(arr - med))
    if mad == 0:
        return arr, 0
    mod_z = 0.6745 * np.abs(arr - med) / mad
    mask = mod_z <= 3.5
    return arr[mask], int((~mask).sum())

DELTA_COLS = ['delta_dist_nm', 'delta_duration_h', 'delta_fuel_kg']
error_rows_clean = []
outlier_log = []

for (adep, ades, clust, ac), grp in df_compared.dropna(subset=DELTA_COLS).groupby(
        ['ADEP', 'ADES', 'cluster_kmeans', _ac_col]):
    row = {'ADEP': adep, 'ADES': ades, 'cluster_kmeans': clust, _ac_col: ac, 'n': len(grp)}
    total_removed = 0
    for col in DELTA_COLS:
        clean, n_rem = _mad_filter(grp[col].values)
        total_removed += n_rem
        if len(clean) >= 2:
            row[f'{col}_mean'] = round(float(np.mean(clean)), 2)
            row[f'{col}_std']  = round(float(np.std(clean, ddof=1)), 2)
            row[f'{col}_p5']   = round(float(np.percentile(clean, 5)), 2)
            row[f'{col}_p95']  = round(float(np.percentile(clean, 95)), 2)
        else:
            for sfx in ('_mean', '_std', '_p5', '_p95'):
                row[f'{col}{sfx}'] = np.nan
    row['n_outliers'] = total_removed
    if total_removed > 0:
        outlier_log.append(
            f'  {adep}-{ades} C{clust} ({ac}): {total_removed} outlier(s) removed from {len(grp)} flights'
        )
    error_rows_clean.append(row)

error_summary = pd.DataFrame(error_rows_clean)

if outlier_log:
    print(f'Outliers removed ({len(outlier_log)} group(s) affected):')
    for msg in outlier_log:
        print(msg)
else:
    print('No outliers detected across any cluster.')

print(f'\nCleaned error_summary: {len(error_summary)} rows')
print('Cluster 3 EGLL-KJFK delta_dist_nm_std (cleaned):',
      error_summary.loc[
          (error_summary['ADEP']=='EGLL') & (error_summary['ADES']=='KJFK') &
          (error_summary['cluster_kmeans']==3), 'delta_dist_nm_std'
      ].values)

ERR_CACHE = '/drive/MyDrive/flight-project/error_summary.csv'
error_summary.to_csv(ERR_CACHE, index=False)
print(f'error_summary cached to Drive ({len(error_summary)} rows)')

# Same MAD-based cleaning applied to the pooled distance/duration error stats (fuel/cost
# are not part of error_summary_pooled -- see cell 36 for why)
DELTA_COLS_POOLED = ['delta_dist_nm_pooled', 'delta_duration_h_pooled']
error_rows_clean_pooled = []
outlier_log_pooled = []

for (adep, ades, clust), grp in df_compared.dropna(subset=DELTA_COLS_POOLED).groupby(
        ['ADEP', 'ADES', 'cluster_kmeans']):
    row = {'ADEP': adep, 'ADES': ades, 'cluster_kmeans': clust, 'n': len(grp)}
    total_removed = 0
    for col in DELTA_COLS_POOLED:
        base = col.replace('_pooled', '')
        clean, n_rem = _mad_filter(grp[col].values)
        total_removed += n_rem
        if len(clean) >= 2:
            row[f'{base}_mean'] = round(float(np.mean(clean)), 2)
            row[f'{base}_std']  = round(float(np.std(clean, ddof=1)), 2)
            row[f'{base}_p5']   = round(float(np.percentile(clean, 5)), 2)
            row[f'{base}_p95']  = round(float(np.percentile(clean, 95)), 2)
        else:
            for sfx in ('_mean', '_std', '_p5', '_p95'):
                row[f'{base}{sfx}'] = np.nan
    row['n_outliers'] = total_removed
    if total_removed > 0:
        outlier_log_pooled.append(
            f'  {adep}-{ades} C{clust} (pooled): {total_removed} outlier(s) removed from {len(grp)} flights'
        )
    error_rows_clean_pooled.append(row)

error_summary_pooled = pd.DataFrame(error_rows_clean_pooled)

if outlier_log_pooled:
    print(f'\nPooled outliers removed ({len(outlier_log_pooled)} group(s) affected):')
    for msg in outlier_log_pooled:
        print(msg)
else:
    print('\nNo pooled outliers detected across any cluster.')

print(f'Cleaned error_summary_pooled: {len(error_summary_pooled)} rows')

ERR_CACHE_POOLED = '/drive/MyDrive/flight-project/error_summary_pooled.csv'
error_summary_pooled.to_csv(ERR_CACHE_POOLED, index=False)
print(f'error_summary_pooled cached to Drive ({len(error_summary_pooled)} rows)')

In [ ]:
# Cell 41b -- cache loader: restore l3_centroids, error_summary, and the pooled
# fallback tables after session restart. Run instead of cells 35-40 when the actual
# vs planned trajectory data is already cached.

L3_CACHE         = '/drive/MyDrive/flight-project/l3_centroids.csv'
L3_CACHE_POOLED  = '/drive/MyDrive/flight-project/l3_centroids_pooled.csv'
ERR_CACHE        = '/drive/MyDrive/flight-project/error_summary.csv'
ERR_CACHE_POOLED = '/drive/MyDrive/flight-project/error_summary_pooled.csv'

try:
    l3_centroids
    error_summary
    l3_centroids_pooled
    error_summary_pooled
    print('l3_centroids, error_summary and pooled fallback tables already in session -- nothing loaded')
except NameError:
    _caches = [L3_CACHE, L3_CACHE_POOLED, ERR_CACHE, ERR_CACHE_POOLED]
    if all(os.path.exists(p) for p in _caches):
        l3_centroids         = pd.read_csv(L3_CACHE)
        l3_centroids_pooled  = pd.read_csv(L3_CACHE_POOLED)
        error_summary        = pd.read_csv(ERR_CACHE)
        error_summary_pooled = pd.read_csv(ERR_CACHE_POOLED)
        _ac_col = next(c for c in l3_centroids.columns if 'AC Type' in c)
        print(f'Loaded from cache:')
        print(f'  l3_centroids:         {len(l3_centroids)} rows')
        print(f'  l3_centroids_pooled:  {len(l3_centroids_pooled)} rows')
        print(f'  error_summary:        {len(error_summary)} rows')
        print(f'  error_summary_pooled: {len(error_summary_pooled)} rows')
    else:
        print('Cache not found -- run cells 35-40 to build and cache first')

In [ ]:
# Cell 41 -- geographic coverage map: all OD pairs coloured by cluster count

import subprocess, importlib
if importlib.util.find_spec('airportsdata') is None:
    subprocess.run(['pip', 'install', '-q', 'airportsdata'], check=True)
import airportsdata
import plotly.graph_objects as go

airports_db = airportsdata.load('ICAO')
_lat = {k: v['lat'] for k, v in airports_db.items()}
_lon = {k: v['lon'] for k, v in airports_db.items()}

od_map = (
    full_summary[['ADEP', 'ADES', 'n_clusters_od']]
    .drop_duplicates()
    .copy()
)
od_map['adep_lat'] = od_map['ADEP'].map(_lat)
od_map['adep_lon'] = od_map['ADEP'].map(_lon)
od_map['ades_lat'] = od_map['ADES'].map(_lat)
od_map['ades_lon'] = od_map['ADES'].map(_lon)
od_map = od_map.dropna(subset=['adep_lat', 'ades_lat']).reset_index(drop=True)

n_total = full_summary[['ADEP', 'ADES']].drop_duplicates().shape[0]
print(f'Pairs with known airport coords: {len(od_map):,} / {n_total:,}')

def _arc_coords(subset):
    lats, lons = [], []
    for _, r in subset.iterrows():
        lats += [r['adep_lat'], r['ades_lat'], None]
        lons += [r['adep_lon'], r['ades_lon'], None]
    return lats, lons

groups = [
    (od_map[od_map['n_clusters_od'] == 1], 'k=1 (single route)',     '#7799BB', 0.12, 0.7),
    (od_map[od_map['n_clusters_od'] == 2], 'k=2 (two variants)',     '#54A0E0', 0.35, 1.2),
    (od_map[od_map['n_clusters_od'] >= 3], 'k>=3 (three+ variants)', '#F58518', 0.48, 1.4),
]

fig = go.Figure()
for subset, name, colour, opacity, width in groups:
    lats, lons = _arc_coords(subset)
    fig.add_trace(go.Scattergeo(
        lat=lats, lon=lons,
        mode='lines',
        line=dict(width=width, color=colour),
        opacity=opacity,
        name=f'{name} -- {len(subset):,} pairs',
        hoverinfo='skip',
    ))

n_alt = od_map[od_map['n_clusters_od'] >= 2].shape[0]
fig.update_geos(
    projection_type='natural earth',
    showland=True,  landcolor='#1a1a2e',
    showocean=True, oceancolor='#16213e',
    showcoastlines=True, coastlinecolor='#444466',
    showframe=False,
    lataxis_range=[-60, 80],
)
fig.update_layout(
    title=dict(
        text=(
            f'Flight route coverage -- Sep 2023<br>'
            f'<sup>{len(od_map):,} O-D pairs | '
            f'{n_alt:,} with route alternatives ({n_alt / len(od_map) * 100:.1f}%)</sup>'
        ),
        x=0.5, xanchor='center', font=dict(color='white'),
    ),
    legend=dict(x=1.0, y=0.0, xanchor='right', yanchor='bottom', bgcolor='rgba(15,15,35,0.7)', font=dict(color='white')),
    paper_bgcolor='#0f0f23',
    height=560,
    margin=dict(t=80, b=10, l=10, r=10),
)
fig.show()


## Methodology

### 1. Objective
Build a route cost and duration simulator for EUROCONTROL's Mercury model. Given an aircraft type and origin-destination airport pair, the simulator returns the historically observed routing variants with predicted cost, duration, fuel, and empirical uncertainty bounds.

---

### 2. Data

| Source | Size | Content |
|--------|------|--------|
| `Flights_20230901_20230930.parquet` | -- | 806,903 scheduled flights, Sep 2023 |
| `Final_Wide_Report.parquet` | -- | Per-flight planned FIR traversal distances |
| `Flight_Points_Filed_20230901_20230930.csv` | 2.1 GB | Filed trajectory waypoints |
| `Flight_Points_Actual_20230901_20230930.csv` | 2.25 GB | Actual trajectory waypoints |
| `Flight_FIRs_Actual_20230901_20230930.csv` | 533 MB | Actual FIR crossing events |

All data covers European airspace (ECAC + North Atlantic) for September 2023.

---

### 3. Three-layer clustering

Flights are organised into three hierarchical layers:

**L1 -- Origin-destination pair.** All flights with the same ADEP-ADES combination form one group. Pairs with fewer than 30 flights are excluded.

**L1.5 -- FIR signature.** Before clustering, flights within each OD pair are grouped by their binary FIR signature: which FIRs were crossed (1/0). This hard grouping ensures that flights with different route topologies are never merged by the distance clustering step.

**L2 -- Routing variant.** Within each (OD pair, FIR signature) group, KMeans is run on the actual FIR distance vector. Optimal k is chosen by silhouette score over k = 2..min(8, n//15). If no k >= 2 achieves a positive silhouette, k = 1 is used.

**L3 -- Aircraft type.** Within each L2 cluster, flights are split by AC type. Cost and fuel burn differ substantially between types on the same corridor, so L3 produces type-specific centroids and error distributions. For the full-dataset predictor (`predict_route_options`), an aircraft-specific route/duration centroid is used only when the queried type has at least `MIN_N_AC` (10) historical flights in that cluster; otherwise it falls back to the pooled (all-aircraft) cluster mean, and the returned `ac_specific` flag records which was used. `query_route_profile` (training pairs only) applies the same threshold, but only pools distance and duration empirically (`l3_centroids_pooled`) -- they vary little by aircraft type on the same corridor. Fuel and cost are not pooled: MTOW-driven fuel burn varies 2-3x across the fleet on one corridor, so a blended historical average is not meaningful (the same reasoning that excludes `delta_cost` from the ac-specific characterisation, section 5). Instead `centroid_fuel_kg`/`centroid_cost_eur` are recomputed via the cost formula (section 4) for the queried aircraft's own MTOW and fuel burn against the pooled cluster's mean FIR distances -- the same approach `predict_route_options` already uses for every query, not just the fallback case. `delta_fuel_kg_*` is therefore not reported when a cluster falls back to the pooled route. The `MIN_N_AC` threshold is not a rare edge case: aggregating `full_summary_ac` by OD pair across the full dataset, a mean of 57% (median 50%) of `(cluster, ac_type)` combinations meet it.

KMeans was chosen over DBSCAN after comparison: DBSCAN produced a high rate of noise-labelled points on binary FIR vectors, while KMeans produced cleaner cluster boundaries with better silhouette scores.

---

### 4. Cost model

**ATC cost** uses the EUROCONTROL service unit formula per FIR:

    SU_i = (d_i / 100) * sqrt(MTOW / 50) * r_i

where d_i is the distance flown in FIR i (km), MTOW is maximum take-off weight (tonnes), and r_i is the Sep 2023 unit rate (EUR/SU). Nav Canada airspace uses a separate weight-distance formula.

**Fuel cost:** fuel_kgh x duration_h x EUR 0.81/kg (Jet-A EIA Sep 2023 average).

**Known limitation:** the planned FIR dataset records both FIR and UIR crossings for the same lateral airspace, inflating planned costs by approximately 30-40%. Cluster costs are valid for relative ranking between routes but not as absolute figures.

---

### 5. Actual vs planned error characterisation

Actual trajectory and FIR data were loaded for 1,434 flights across three training OD pairs (EGLL-KJFK, LEBL-LEPA, LPPT-EDDB). Three per-flight deltas were computed:

- **Delta_dist (nm):** actual haversine distance minus planned haversine distance
- **Delta_duration (h):** actual block time minus planned block time
- **Delta_fuel (kg):** derived from Delta_duration x fuel_kgh

Delta_cost is excluded: even after applying the FIR/UIR deduplication fix (see section 4), cost depends on MTOW and ATC rates which introduce additional uncertainty not present in the trajectory metrics. Distance and duration are more direct observables.

Outliers are removed using the modified z-score method (Iglewicz & Hoaglin, threshold 3.5) before computing error statistics. This is particularly important for small L3 groups (n < 10) where a single outlier flight can dominate the distribution. Per-cluster error statistics are reported as mean, standard deviation, and 5th/95th percentiles.

**Key findings:**
- Delta_dist mean is near zero for all clusters -- actual tracks closely follow filed plans
- LEBL-LEPA Delta_dist mean approx. -12 nm -- short-haul routes benefit from ATC direct routings
- Delta_duration mean approx. 0 +/- 0.05 h across all pairs -- filed block time is a reliable point estimate
- Error characterisation is only validated on the three training pairs

---

### 6. Full-dataset extension

The clustering and cost pipeline was applied to all 7,345 qualifying OD pairs (639,192 flights):

| Metric | Value |
|--------|-------|
| OD pairs processed | 7,345 |
| Pairs with k >= 2 (routing alternatives) | 3,383 (46.1%) |
| Total cluster rows in full_summary | 12,751 |
| MTOW coverage | 96.9% (45 AC types) |

The 30-50 flights/pair band has 21% k >= 2; this rises to 51-75% for pairs with 50+ flights. The threshold of 30 was retained: k = 1 pairs in this band are valid (they indicate no routing variant exists), and discarding them would reduce dataset coverage without improving accuracy.

---

### 7. Out-of-sample validation

EGLL-LGAV (London Heathrow to Athens) was held out from the training sample and clustered independently. The clustering produced 5 route variants across 225 flights, confirming the pipeline generalises to unseen OD pairs.

The formula-based cost prediction was evaluated against per-flight actual costs:

| Metric | Value |
|--------|-------|
| n (flights) | 225 |
| Clusters found | 5 |
| MAE | EUR 381 |
| Mean actual cost | EUR 10,249 |
| Relative MAE | 3.7% |
| Mean bias | +EUR 126 (1.2% overestimation) |
| R-squared | 0.02 |

The MAE of 3.7% is sufficient for route planning purposes. R-squared is low because the model assigns one predicted cost per cluster -- it captures between-cluster differences but not within-cluster flight-to-flight variance driven by load factor, weather, and actual ATC routing. This is expected behaviour for a cluster-level predictor.

Note: two FIRs on this route (Albania LAAAFIR, Serbia LYBAUIR) previously used estimated unit rates; these are now verified against the Eurocontrol CRCO Sep 2023 monthly adjusted unit rate table.

---

### 8. Known limitations

- Error characterisation covers three training OD pairs only. To extend to additional pairs, load actual FIR and trajectory data for those flights and re-run cells 34-40. The query function returns centroid-only with a clear note when error stats are unavailable.
- L3 clusters with n_l3 < 10 have unreliable error bounds (flagged in `query_route_profile`)
- Unknown AC types (not in `MTOW_TONNES` / `FUEL_KGH`) still raise `ValueError` -- extend those dicts to resolve. This is different from an aircraft with too few flights in a cluster, which now falls back to the pooled cluster mean instead of failing (see section 3).
- `query_route_profile` now shares `predict_route_options`'s pooled-centroid fallback (own `l3_centroids_pooled` / `error_summary_pooled` tables, same `MIN_N_AC` threshold) -- it only raises `ValueError` when the OD pair itself has no L3 data at all, not when a specific aircraft type is missing from one of its clusters
- FIR+UIR double-counting in the source data has been corrected in the cost pipeline: when both XXXXXFIR and XXXXXUIR appear for the same ANSP, only one is charged. This reduces planned cost inflation vs actual.
- Cluster count is bounded at k = 8; very high-frequency pairs with more than eight genuine routing variants are truncated
